# claude haiku code 

In [2]:
import base64
import json
import os
import re
from collections import defaultdict
from typing import Any
from json import JSONDecoder

import pandas as pd
import boto3
from pdf2image import convert_from_path

EXTRACTION_SCHEMA = {
  "claim_number": "",
  "tax_id": "",
  "diagnosis_codes": [],
  "info": [
      {
          "date_of_service": "",
          "cpt_codes": "",
          "charges": "",
          "units": ""
      }
  ]
}

ARRAY_FIELDS = {"diagnosis_codes", "info"}
SCALAR_FIELDS = [
    "claim_number",
    "tax_id"
]
OBJECT_FIELDS = []


In [3]:
from dotenv import load_dotenv
import os

# Load AWS credentials from .env file
load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION", "ap-south-1")

In [4]:
def get_bedrock_client():
    """Create and return AWS Bedrock runtime client."""
    session = boto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        aws_session_token=AWS_SESSION_TOKEN,
        region_name=AWS_DEFAULT_REGION,
    )
    return session.client("bedrock-runtime")


# Test connection
try:
    client = get_bedrock_client()
    print("✓ Bedrock client initialized successfully")
except Exception as e:
    print(f"✗ Error initializing Bedrock client: {e}")

✓ Bedrock client initialized successfully


In [5]:
fields_to_extract="Claim Number,Tax Id,Diagnosis Codes,Info(Date Of Service,CPT Codes,Charges,Units)"

response_format=r'''{
  "claim_number": "",
  "tax_id": "",
  "diagnosis_codes": [],
  "info": [
      {
          "date_of_service": "",
          "cpt_codes": "",
          "charges": "",
          "units": ""
      }
  ]
}'''

system_prompt = """You are an AI assistant specialized in data extraction and mapping. Describe every detail you can about this image by thoroughly identifying and organizing the data. Use only the information present within the document itself. Note: Do not extract double quotes or single quotes from the image provided for you. """

EXTRACTION_PROMPT = f"""{system_prompt}

extract {fields_to_extract} if present give the output in JSON format,follow the schema: 
{response_format}"""


In [6]:
def page_suffix(index: int) -> str:
    """Convert page index to letter suffix (1->a, 2->b, ..., 27->aa)"""
    letters = ""
    n = index
    while n > 0:
        n, rem = divmod(n - 1, 26)
        letters = chr(97 + rem) + letters
    return letters


def build_page_document_name(doc_name: str, page_number: int) -> str:
    """Build document name with page suffix for multi-page documents"""
    if page_number == 1:
        return doc_name
    return f"{doc_name}_{page_suffix(page_number)}"


def get_images_from_folder(images_folder: str) -> dict[str, list[str]]:
    """
    Scan images_folder for subdirectories containing images.
    Returns: {document_name: [path1, path2, ...]}
    """
    images_by_document = defaultdict(list)
    
    for item in os.listdir(images_folder):
        item_path = os.path.join(images_folder, item)
        
        # Handle image files directly in folder
        if os.path.isfile(item_path) and item.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            doc_name = os.path.splitext(item)[0]
            images_by_document[doc_name].append(item_path)
        
        # Handle subdirectories with images
        elif os.path.isdir(item_path):
            for img_file in sorted(os.listdir(item_path)):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    img_path = os.path.join(item_path, img_file)
                    images_by_document[item].append(img_path)
    
    # Sort image lists by filename
    for doc in images_by_document:
        images_by_document[doc].sort()
    
    return dict(images_by_document)

In [7]:
def ask_about_images_json(client, image_paths: list[str], prompt: str) -> str:
    """
    Send images to Claude via Bedrock Converse API and get JSON response.

    Args:
        client: Bedrock runtime client
        image_paths: List of image file paths
        prompt: Extraction prompt

    Returns:
        Raw response text from Claude
    """
    content = []

    # Add images using Bedrock Converse message content schema
    for image_path in image_paths:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()
        # Bedrock expects image format like jpeg/png/webp (not MIME type).
        image_format = "jpeg" if ext == "jpg" else ext
        if image_format not in {"jpeg", "png", "webp", "gif"}:
            image_format = "jpeg"

        content.append({
            "image": {
                "format": image_format,
                "source": {"bytes": image_bytes},
            }
        })

    # Add text prompt
    content.append({"text": prompt})

    # Call Claude API
    response = client.converse(
        modelId="anthropic.claude-3-haiku-20240307-v1:0",
        messages=[{"role": "user", "content": content}],
        inferenceConfig={
            "maxTokens": 1400,
            "temperature": 0,
        },
    )

    # Extract response text
    message_content = response["output"]["message"]["content"]

    if isinstance(message_content, str):
        return message_content
    elif isinstance(message_content, list):
        text_parts = []
        for item in message_content:
            if isinstance(item, dict) and "text" in item:
                text_parts.append(item.get("text", ""))
        return "\n".join(text_parts)
    else:
        return str(message_content)

In [8]:
def parse_model_json(text: str) -> dict[str, Any]:
    """
    Parse JSON response from Claude with 4-stage fallback strategy.

    Stage 1: Direct JSON parse
    Stage 2: Strip markdown code fences
    Stage 3: Scan for first '{' and parse from there
    Stage 4: Fail with error
    """
    decoder = JSONDecoder()

    # Stage 1: Direct parse
    try:
        result = json.loads(text)
        if isinstance(result, dict):
            return result
    except Exception:
        pass

    # Stage 2: Strip markdown code fences
    if text.startswith("```") and text.endswith("```"):
        try:
            cleaned = "\n".join(text.split("\n")[1:-1]).strip()
            result = json.loads(cleaned)
            if isinstance(result, dict):
                return result
        except Exception:
            pass

    # Stage 3: Scan for first '{' and raw_decode
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                result, _ = decoder.raw_decode(text[i:])
                if isinstance(result, dict):
                    return result
            except Exception:
                pass

    # Stage 4: Failed
    raise ValueError(f"Could not parse JSON from response: {text[:200]}...")


def extract_partial_fields(raw_text: str) -> dict[str, Any]:
    """
    Fallback extraction using regex when JSON parsing fails.
    Attempts to extract fields using pattern matching.
    """
    partial = EXTRACTION_SCHEMA.copy()

    # Extract scalar fields
    for key in SCALAR_FIELDS:
        # Try quoted value pattern
        pattern = rf'"{key}"\s*:\s*"(.*?)"'
        match = re.search(pattern, raw_text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            partial[key] = match.group(1).strip()
            continue

        # Try unquoted value pattern
        pattern = rf"{key}\s*:\s*([^,}}]+)"
        match = re.search(pattern, raw_text, flags=re.IGNORECASE)
        if match:
            partial[key] = match.group(1).strip().strip('"').strip("'")

    # Extract array fields
    for key in ARRAY_FIELDS:
        pattern = rf'"{key}"\s*:\s*\[(.*?)\]'
        match = re.search(pattern, raw_text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            items = []
            for part in match.group(1).split(","):
                value = part.strip().strip('"').strip("'")
                if value and value.lower() != "none":
                    items.append(value)
            if items:
                partial[key] = items

    return partial

In [9]:
def flatten_for_excel(data: dict[str, Any]) -> dict[str, Any]:
    """Convert nested result dict to flat row for Excel export."""
    row = {
        "document_name": data.get("document_name", ""),
        "page_number": data.get("page_number", ""),
        "parse_status": data.get("parse_status", ""),
    }
    
    # Add extracted fields
    extracted = data.get("extracted_data", {})
    for key in EXTRACTION_SCHEMA.keys():
        value = extracted.get(key, "")
        # Convert arrays to pipe-delimited strings for Excel
        if isinstance(value, list):
            row[key] = " | ".join(str(v) for v in value)
        else:
            row[key] = value
    
    return row


def process_documents_to_excel(client, images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents.
    
    Args:
        client: Bedrock runtime client
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print(f"Starting OCR extraction for Claude 3 Haiku")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")
    
    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")
    
    all_results = []
    failed_pages = []
    
    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")
        
        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json(client, [image_path], EXTRACTION_PROMPT)
                
                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception as e:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"
                
                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }
                
                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)
                
                print(f"  Page {page_idx}: {parse_status} ✓")
                
            except Exception as e:
                print(f"  Page {page_idx}: ERROR - {str(e)[:50]}")
                failed_pages.append({
                    "document": doc_name,
                    "page_number": page_idx,
                    "image_file": os.path.basename(image_path),
                    "error": str(e),
                })
    
    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)
        
        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]
        
        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")
        
        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")
        
        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")




def clean_gt_schema(data):
    """Recursively strips values to create a blank JSON template, completely flattening 'ocr_output' fields."""
    if isinstance(data, dict):
        # If this dict represents a leaf node with ocr_output, flatten it to just a string
        if "ocr_output" in data:
            return ""
        # Otherwise, recursively clean the nested dict, stripping confidence if present
        return {k: clean_gt_schema(v) for k, v in data.items() if k != "confidence"}
    elif isinstance(data, list):
        if len(data) > 0:
            return [clean_gt_schema(data[0])]
        return []
    else:
        return ""

def process_pdf_folders(client_func, root_folder: str, client_arg=None, model_suffix="") -> None:
    """
    Traverses root_folder looking for subfolders containing .pdf files.
    Finds the corresponding Ground Truth JSON, creates a dynamic schema from it,
    converts PDF to images, and extracts data via the provided client_func.
    """
    print(f"\n{'='*60}")
    print(f"Starting Dynamic OCR extraction pipeline {model_suffix}")
    print(f"Root folder: {root_folder}")
    print(f"{'='*60}\n")
    
    pdf_tasks = []
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_path = os.path.join(root, file)
                pdf_tasks.append((root, file, pdf_path))
                
    print(f"Found {len(pdf_tasks)} PDFs to process.\n")
    
    for idx, (folder, pdf_name, pdf_path) in enumerate(pdf_tasks, 1):
        print(f"[{idx}/{len(pdf_tasks)}] Processing: {pdf_path}")
        
        try:
            # 1. Load Ground Truth to get Dynamic Schema
            gt_name = os.path.splitext(pdf_name)[0] + ".json"
            gt_path = os.path.join(folder, gt_name)
            
            if not os.path.exists(gt_path):
                print(f"  ✗ Ground truth not found: {gt_name}. Skipping...")
                continue
                
            with open(gt_path, 'r', encoding='utf-8') as f:
                gt_data = json.load(f)
                
            # Create a blank template stripped of actual values, completely flattened
            dynamic_schema = clean_gt_schema(gt_data)
            
            dynamic_prompt = f"""You are an AI assistant specialized in data extraction and mapping. 
Describe every detail you can about the provided document images. 
Extract the information to match the following JSON schema exactly.
Populate the empty strings with the extracted data. Do not extract double quotes or single quotes from the image.

{json.dumps(dynamic_schema, indent=2)}"""

            # 2. Convert PDF to images
            from pdf2image import convert_from_path
            images = convert_from_path(pdf_path, poppler_path=r"C:\poppler\Library\bin")
            if not images:
                print(f"  ✗ No images extracted from PDF")
                continue
            
            temp_image_paths = []
            for i, img in enumerate(images):
                temp_path = os.path.join(folder, f"temp_page_{i}.jpg")
                img.save(temp_path, "JPEG")
                temp_image_paths.append(temp_path)
            
            print(f"  ✓ Converted PDF to {len(temp_image_paths)} images")
            
            # 3. Extract data from images using the Dynamic Prompt
            if client_arg:
                raw_response = client_func(client_arg, temp_image_paths, dynamic_prompt)
            else:
                raw_response = client_func(temp_image_paths, dynamic_prompt)
                
            # 4. Parse JSON
            try:
                extracted_data = parse_model_json(raw_response)
                status = "Success"
            except Exception as e:
                extracted_data = {"error": f"JSON parsing failed: {str(e)}", "raw_response": raw_response}
                status = "Failed parsing"
                
            # 5. Save JSON to same folder
            out_json_name = f"extracted_output{'_' + model_suffix if model_suffix else ''}.json"
            out_json_path = os.path.join(folder, out_json_name)
            with open(out_json_path, 'w', encoding='utf-8') as f:
                json.dump(extracted_data, f, indent=2)
                
            print(f"  ✓ Saved extraction to: {out_json_name} ({status})")
            
            # Cleanup temp images
            for p in temp_image_paths:
                if os.path.exists(p):
                    os.remove(p)
                    
        except Exception as e:
            print(f"  ✗ ERROR processing {pdf_name}: {str(e)}")
            import traceback
            traceback.print_exc()


In [37]:
# Configure root path containing PDFs
ROOT_PDF_FOLDER = r"C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images" # Update as needed

# Run extraction for Claude
try:
    process_pdf_folders(ask_about_images_json, ROOT_PDF_FOLDER, client_arg=client, model_suffix="claude")
except Exception as e:
    print(f"✗ Error during processing: {e}")
    import traceback
    traceback.print_exc()



Starting OCR extraction for Claude 3 Haiku
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_claude_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[12/28] P

# LLama-Maverick Implementation 

In [38]:
# Load Llama credentials
from dotenv import load_dotenv
# Load AWS credentials from .env file
load_dotenv(override=True)
llama_endpoint = os.getenv("llama_endpoint")
llama_deployment_name = os.getenv("llama_deployment_name")
llama_api_key = os.getenv("llama_api_key")

print("Llama Credentials Loaded:")
print(f"✓ Endpoint: {llama_endpoint}")
print(f"✓ Deployment: {llama_deployment_name}")
print(f"✓ API Key: {llama_api_key[:20]}..." if llama_api_key else "✗ API Key: NOT FOUND")


Llama Credentials Loaded:
✓ Endpoint: https://intelligenttechsop.services.ai.azure.com/openai/v1
✓ Deployment: Llama-4-Maverick-17B-128E-Instruct-FP8
✓ API Key: 4g3rTLcQStZweayjG3Vb...


In [39]:
import requests


def ask_about_images_json_llama(image_paths: list[str], prompt: str) -> str:
    """
    Send images to Llama-Maverick endpoint and get JSON response text.

    Args:
        image_paths: List of image file paths
        prompt: Extraction prompt

    Returns:
        Raw response text from model
    """
    if not all([llama_endpoint, llama_deployment_name, llama_api_key]):
        raise ValueError("Missing Llama credentials (endpoint/deployment/api_key)")

    content = []

    # Add images as data URLs (same multi-modal pattern as Claude pipeline).
    for image_path in image_paths:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()
        mime_type = "image/jpeg" if ext == "jpg" else f"image/{ext}"
        if ext not in {"jpg", "jpeg", "png", "webp", "gif"}:
            mime_type = "image/jpeg"

        image_b64 = base64.b64encode(image_bytes).decode("utf-8")
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime_type};base64,{image_b64}"},
            }
        )

    # Add text prompt
    content.append({"type": "text", "text": prompt})

    headers = {
        "Content-Type": "application/json",
        "api-key": llama_api_key,
    }

    payload = {
        "model": llama_deployment_name,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "max_tokens": 1400,
    }

    response = requests.post(
        f"{llama_endpoint}/chat/completions",
        headers=headers,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()

    data = response.json()
    return data["choices"][0]["message"]["content"]

In [40]:
def process_documents_to_excel_llama(images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents with Llama-Maverick.

    Args:
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print("Starting OCR extraction for Llama-Maverick")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")

    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")

    all_results = []
    failed_pages = []

    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")

        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json_llama([image_path], EXTRACTION_PROMPT)

                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"

                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }

                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)

                print(f"  Page {page_idx}: {parse_status} ✓")

            except Exception as e:
                print(f"  Page {page_idx}: ERROR - {str(e)[:50]}")
                failed_pages.append(
                    {
                        "document": doc_name,
                        "page_number": page_idx,
                        "image_file": os.path.basename(image_path),
                        "error": str(e),
                    }
                )

    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)

        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]

        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")

        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")

        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")

In [41]:
# Run extraction for Llama
try:
    process_pdf_folders(ask_about_images_json_llama, ROOT_PDF_FOLDER, model_suffix="llama")
except Exception as e:
    print(f"✗ Error during processing: {e}")



Starting OCR extraction for Llama-Maverick
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_llama_maverick_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[

# Chatgpt 5.4

In [42]:
# Load ChatGPT credentials
from dotenv import load_dotenv
# Load credentials from .env file
load_dotenv(override=True)
chatgpt_endpoint = os.getenv("chatgpt_endpoint")
chatgpt_deployment_name = os.getenv("chatgpt_deployment_name")
chatgpt_api_key = os.getenv("chatgpt_api_key")
chatgpt_api_version = os.getenv("chatgpt_api_version", "2025-01-01-preview")

print("ChatGPT Credentials Loaded:")
print(f"✓ Endpoint: {chatgpt_endpoint}")
print(f"✓ Deployment: {chatgpt_deployment_name}")
print(f"✓ API Version: {chatgpt_api_version}")
print(f"✓ API Key: {chatgpt_api_key[:20]}..." if chatgpt_api_key else "✗ API Key: NOT FOUND")


ChatGPT Credentials Loaded:
✓ Endpoint: https://sagility-bt-instance-1.openai.azure.com/
✓ Deployment: gpt-5.4
✓ API Version: 2025-01-01-preview
✓ API Key: CHoDdvl0AN0tWPw93K2b...


In [43]:
import requests


def ask_about_images_json_chatgpt(image_paths: list[str], prompt: str) -> str:
    """
    Send images to ChatGPT 5.4 endpoint and get JSON response text.

    Supports Azure endpoint styles:
    1) Resource root: https://<resource>.openai.azure.com
    2) Deployment path: .../openai/deployments/<deployment>
    3) v1 path: .../openai/v1
    """
    if not all([chatgpt_endpoint, chatgpt_deployment_name, chatgpt_api_key]):
        raise ValueError("Missing ChatGPT credentials (endpoint/deployment/api_key)")

    base_endpoint = chatgpt_endpoint.rstrip("/")

    # Build request URL based on endpoint style.
    if "/openai/deployments/" in base_endpoint:
        request_url = f"{base_endpoint}/chat/completions?api-version={chatgpt_api_version}"
        include_model_in_payload = False
    elif base_endpoint.endswith("/openai/v1"):
        request_url = f"{base_endpoint}/chat/completions"
        include_model_in_payload = True
    else:
        request_url = (
            f"{base_endpoint}/openai/deployments/{chatgpt_deployment_name}"
            f"/chat/completions?api-version={chatgpt_api_version}"
        )
        include_model_in_payload = False

    content = []

    # Add images as data URLs (same multi-modal pattern as other pipelines).
    for image_path in image_paths:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()
        mime_type = "image/jpeg" if ext == "jpg" else f"image/{ext}"
        if ext not in {"jpg", "jpeg", "png", "webp", "gif"}:
            mime_type = "image/jpeg"

        image_b64 = base64.b64encode(image_bytes).decode("utf-8")
        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{image_b64}",
                    "detail": "high",
                },
            }
        )

    # Add text prompt
    content.append({"type": "text", "text": prompt})

    headers = {
        "Content-Type": "application/json",
        "api-key": chatgpt_api_key,
    }

    base_payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
    }
    if include_model_in_payload:
        base_payload["model"] = chatgpt_deployment_name

    # Try compatible payload variants because Azure/model versions differ on token parameter names.
    payload_attempts = [
        {**base_payload, "response_format": {"type": "json_object"}, "max_completion_tokens": 1400},
        {**base_payload, "max_completion_tokens": 1400},
        {**base_payload, "max_tokens": 1400},
    ]

    last_error = None
    for payload in payload_attempts:
        response = requests.post(
            request_url,
            headers=headers,
            json=payload,
            timeout=120,
        )
        if response.ok:
            data = response.json()
            return data["choices"][0]["message"]["content"]
        last_error = f"HTTP {response.status_code}: {response.text[:1000]}"

    raise RuntimeError(f"ChatGPT request failed for {request_url}. Last error: {last_error}")

In [44]:
def process_documents_to_excel_chatgpt(images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents with ChatGPT 5.4.

    Args:
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print("Starting OCR extraction for ChatGPT 5.4")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")

    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")

    all_results = []
    failed_pages = []

    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")

        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json_chatgpt([image_path], EXTRACTION_PROMPT)

                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"

                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }

                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)

                print(f"  Page {page_idx}: {parse_status} ✓")

            except Exception as e:
                print(f"  Page {page_idx}: ERROR - {str(e)[:50]}")
                failed_pages.append(
                    {
                        "document": doc_name,
                        "page_number": page_idx,
                        "image_file": os.path.basename(image_path),
                        "error": str(e),
                    }
                )

    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)

        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]

        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")

        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")

        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")

In [45]:
# Run extraction for ChatGPT 5.4
try:
    process_pdf_folders(ask_about_images_json_chatgpt, ROOT_PDF_FOLDER, model_suffix="chatgpt")
except Exception as e:
    print(f"✗ Error during processing: {e}")



Starting OCR extraction for ChatGPT 5.4
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt54_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[12/28] P

# chatgpt 5.4 nano

In [46]:
# Load ChatGPT 5.4 Nano credentials
from dotenv import load_dotenv
# Load credentials from .env file
load_dotenv(override=True)
chatgpt_nano_endpoint = os.getenv("chatgpt_endpoint_nano")
chatgpt_nano_deployment_name = os.getenv("chatgpt_deployment_name_nano")
chatgpt_nano_api_key = os.getenv("chatgpt_api_key_nano")
chatgpt_nano_api_version = os.getenv("chatgpt_api_version_nano", "2025-01-01-preview")

print("ChatGPT 5.4 Nano Credentials Loaded:")
print(f"✓ Endpoint: {chatgpt_nano_endpoint}")
print(f"✓ Deployment: {chatgpt_nano_deployment_name}")
print(f"✓ API Version: {chatgpt_nano_api_version}")
print(f"✓ API Key: {chatgpt_nano_api_key[:20]}..." if chatgpt_nano_api_key else "✗ API Key: NOT FOUND")


ChatGPT 5.4 Nano Credentials Loaded:
✓ Endpoint: https://sagility-bt-instance-1.openai.azure.com/
✓ Deployment: gpt-5.4-nano
✓ API Version: 2025-01-01-preview
✓ API Key: CHoDdvl0AN0tWPw93K2b...


In [47]:
import requests


def ask_about_images_json_chatgpt_nano(image_paths: list[str], prompt: str) -> str:
    """
    Send images to ChatGPT 5.4 Nano endpoint and get JSON response text.

    Supports Azure endpoint styles:
    1) Resource root: https://<resource>.openai.azure.com
    2) Deployment path: .../openai/deployments/<deployment>
    3) v1 path: .../openai/v1
    """
    if not all([chatgpt_nano_endpoint, chatgpt_nano_deployment_name, chatgpt_nano_api_key]):
        raise ValueError("Missing ChatGPT Nano credentials (endpoint/deployment/api_key)")

    base_endpoint = chatgpt_nano_endpoint.rstrip("/")

    # Build request URL based on endpoint style.
    if "/openai/deployments/" in base_endpoint:
        request_url = f"{base_endpoint}/chat/completions?api-version={chatgpt_nano_api_version}"
        include_model_in_payload = False
    elif base_endpoint.endswith("/openai/v1"):
        request_url = f"{base_endpoint}/chat/completions"
        include_model_in_payload = True
    else:
        request_url = (
            f"{base_endpoint}/openai/deployments/{chatgpt_nano_deployment_name}"
            f"/chat/completions?api-version={chatgpt_nano_api_version}"
        )
        include_model_in_payload = False

    content = []

    # Add images as data URLs (same multi-modal pattern as other pipelines).
    for image_path in image_paths:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()
        mime_type = "image/jpeg" if ext == "jpg" else f"image/{ext}"
        if ext not in {"jpg", "jpeg", "png", "webp", "gif"}:
            mime_type = "image/jpeg"

        image_b64 = base64.b64encode(image_bytes).decode("utf-8")
        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{image_b64}",
                    "detail": "high",
                },
            }
        )

    # Add text prompt
    content.append({"type": "text", "text": prompt})

    headers = {
        "Content-Type": "application/json",
        "api-key": chatgpt_nano_api_key,
    }

    base_payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
    }
    if include_model_in_payload:
        base_payload["model"] = chatgpt_nano_deployment_name

    # Try compatible payload variants because Azure/model versions differ on token parameter names.
    payload_attempts = [
        {**base_payload, "response_format": {"type": "json_object"}, "max_completion_tokens": 1400},
        {**base_payload, "max_completion_tokens": 1400},
        {**base_payload, "max_tokens": 1400},
    ]

    last_error = None
    for payload in payload_attempts:
        response = requests.post(
            request_url,
            headers=headers,
            json=payload,
            timeout=120,
        )
        if response.ok:
            data = response.json()
            return data["choices"][0]["message"]["content"]
        last_error = f"HTTP {response.status_code}: {response.text[:1000]}"

    raise RuntimeError(f"ChatGPT Nano request failed for {request_url}. Last error: {last_error}")

In [48]:
# Run extraction for ChatGPT Nano
try:
    process_pdf_folders(ask_about_images_json_chatgpt_nano, ROOT_PDF_FOLDER, model_suffix="chatgpt_nano")
except Exception as e:
    pass


In [49]:
# Run extraction for ChatGPT Nano
try:
    process_pdf_folders(ask_about_images_json_chatgpt_nano, ROOT_PDF_FOLDER, model_suffix="chatgpt_nano")
except Exception as e:
    pass



Starting OCR extraction for ChatGPT 5.4 Nano
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt54_nano_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓

# llama_scout


In [50]:
# Load ChatGPT credentials
from dotenv import load_dotenv
# Load credentials from .env file
load_dotenv(override=True)
llama_scout_endpoint = os.getenv("llama_scout_endpoint")
llama_scout_deployment_name = os.getenv("llama_scout_deployment_name")
llama_scout_api_key = os.getenv("llama_scout_api_key")
#llama_scout_api_version = os.getenv("llama_scout_api_version", "2025-01-01-preview")

print("Llama Scout Credentials Loaded:")
print(f"✓ Endpoint: {llama_scout_endpoint}")
print(f"✓ Deployment: {llama_scout_deployment_name}")
#print(f"✓ API Version: {llama_scout_api_version}")
print(f"✓ API Key: {llama_scout_api_key[:20]}..." if llama_scout_api_key else "✗ API Key: NOT FOUND")


Llama Scout Credentials Loaded:
✓ Endpoint: https://sagility-extraction-resource.openai.azure.com/openai/v1
✓ Deployment: Llama-4-Scout-17B-16E-Instruct
✓ API Key: IJFLE9tdZO8LUekTV5Kj...


In [51]:
import requests


def ask_about_images_json_llama_scout(image_paths: list[str], prompt: str) -> str:
    """
    Send images to Llama Scout endpoint and get JSON response text.

    Args:
        image_paths: List of image file paths
        prompt: Extraction prompt

    Returns:
        Raw response text from model
    """
    if not all([llama_scout_endpoint, llama_scout_deployment_name, llama_scout_api_key]):
        raise ValueError("Missing Llama Scout credentials (endpoint/deployment/api_key)")

    content = []

    # Add images as data URLs (same multi-modal pattern as other pipelines).
    for image_path in image_paths:
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()
        mime_type = "image/jpeg" if ext == "jpg" else f"image/{ext}"
        if ext not in {"jpg", "jpeg", "png", "webp", "gif"}:
            mime_type = "image/jpeg"

        image_b64 = base64.b64encode(image_bytes).decode("utf-8")
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime_type};base64,{image_b64}"},
            }
        )

    # Add text prompt
    content.append({"type": "text", "text": prompt})

    headers = {
        "Content-Type": "application/json",
        "api-key": llama_scout_api_key,
    }

    payload = {
        "model": llama_scout_deployment_name,
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
        "max_tokens": 1400,
    }

    response = requests.post(
        f"{llama_scout_endpoint}/chat/completions",
        headers=headers,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()

    data = response.json()
    return data["choices"][0]["message"]["content"]

In [52]:
def process_documents_to_excel_llama_scout(images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents with Llama Scout.

    Args:
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print("Starting OCR extraction for Llama Scout")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")

    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")

    all_results = []
    failed_pages = []

    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")

        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json_llama_scout([image_path], EXTRACTION_PROMPT)

                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"

                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }

                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)

                print(f"  Page {page_idx}: {parse_status} ✓")

            except Exception as e:
                print(f"  Page {page_idx}: ERROR - {str(e)[:50]}")
                failed_pages.append(
                    {
                        "document": doc_name,
                        "page_number": page_idx,
                        "image_file": os.path.basename(image_path),
                        "error": str(e),
                    }
                )

    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)

        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]

        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")

        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")

        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")

In [53]:
# Configure paths
IMAGES_FOLDER_LLAMA_SCOUT = r"C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images"
OUTPUT_EXCEL_LLAMA_SCOUT = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_llama_scout_doc5.xlsx"


# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_EXCEL_LLAMA_SCOUT), exist_ok=True)

# Run extraction
try:
    process_documents_to_excel_llama_scout(IMAGES_FOLDER_LLAMA_SCOUT, OUTPUT_EXCEL_LLAMA_SCOUT)
except Exception as e:
    print(f"✗ Error during processing: {e}")
    import traceback
    traceback.print_exc()


Starting OCR extraction for Llama Scout
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_llama_scout_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[12/28]

# Chatgpt 5.5 

In [43]:
# Load ChatGPT credentials
from dotenv import load_dotenv
import os
from openai import AzureOpenAI
import base64
# Load credentials from .env file
load_dotenv(override=True)
chatgpt_endpoint = os.getenv("AZURE_ENDPOINT")
chatgpt_deployment_name = os.getenv("AZURE_GT_DEPLOYMENT")
chatgpt_api_key = os.getenv("AZURE_API_KEY")
chatgpt_api_version = os.getenv("AZURE_API_VERSION", "2025-04-01-preview")

print("ChatGPT Credentials Loaded:")
print(f"✓ Endpoint: {chatgpt_endpoint}")
print(f"✓ Deployment: {chatgpt_deployment_name}")
print(f"✓ API Version: {chatgpt_api_version}")
print(f"✓ API Key: {chatgpt_api_key[:20]}..." if chatgpt_api_key else "✗ API Key: NOT FOUND")


ChatGPT Credentials Loaded:
✓ Endpoint: https://mauli-m78wc5qg-eastus2.cognitiveservices.azure.com/
✓ Deployment: gpt-5.5
✓ API Version: 2025-04-01-preview
✓ API Key: BrJXmiO3HmuI2k4GXhmi...


In [44]:
azure_client = AzureOpenAI(
    azure_endpoint=chatgpt_endpoint,
    api_key=chatgpt_api_key,
    api_version=chatgpt_api_version
)

In [45]:
import requests


def ask_about_images_json_chatgpt(image_paths: list[str], prompt: str) -> str:
    """
    Send images to Azure OpenAI GPT-5.5 and return raw response text.
    """

    content = []

    # Add images
    for image_path in image_paths:

        with open(image_path, "rb") as f:
            image_bytes = f.read()

        ext = image_path.split(".")[-1].lower()

        mime_map = {
            "png": "image/png",
            "jpg": "image/jpeg",
            "jpeg": "image/jpeg",
            "webp": "image/webp",
            "bmp": "image/bmp",
            "gif": "image/gif"
        }

        mime_type = mime_map.get(ext, "image/png")

        image_b64 = base64.b64encode(image_bytes).decode("utf-8")

        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{image_b64}",
                    "detail": "high"
                }
            }
        )

    # Add prompt
    content.append(
        {
            "type": "text",
            "text": prompt
        }
    )

    try:

        response = azure_client.chat.completions.create(
            model=chatgpt_deployment_name,
            messages=[
                {
                    "role": "user",
                    "content": content
                }
            ],
            max_completion_tokens=1400
        )

        return response.choices[0].message.content

    except Exception as e:
        raise RuntimeError(f"Azure OpenAI request failed: {e}")    # Add text prompt
    content.append({"type": "text", "text": prompt})

    headers = {
        "Content-Type": "application/json",
        "api-key": chatgpt_api_key,
    }

    base_payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": 0,
    }
    if include_model_in_payload:
        base_payload["model"] = chatgpt_deployment_name

    # Try compatible payload variants because Azure/model versions differ on token parameter names.
    payload_attempts = [
    {**base_payload, "response_format": {"type": "json_object"}, "max_completion_tokens": 1400},
    {**base_payload, "max_completion_tokens": 1400},
    ]

    last_error = None
    for payload in payload_attempts:
        print("\n====================")
        print("PAYLOAD BEING TRIED")
        print(payload)
        print("====================")

        response = requests.post(
            request_url,
            headers=headers,
            json=payload,
            timeout=120,
        )

        print("STATUS:", response.status_code)
        print("RESPONSE:")
        print(response.text)
        if response.ok:
            data = response.json()
            return data["choices"][0]["message"]["content"]
        last_error = f"HTTP {response.status_code}: {response.text[:1000]}"

    raise RuntimeError(f"ChatGPT request failed for {request_url}. Last error: {last_error}")

In [13]:
def process_documents_to_excel_chatgpt(images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents with ChatGPT 5.5.

    Args:
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print("Starting OCR extraction for ChatGPT 5.5")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")

    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")

    all_results = []
    failed_pages = []

    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")

        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json_chatgpt([image_path], EXTRACTION_PROMPT)

                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"

                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }

                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)

                print(f"  Page {page_idx}: {parse_status} ✓")

            except Exception as e:
                print(f"\nPage {page_idx} FULL ERROR:")
                print(str(e))

                failed_pages.append(
                    {
                        "document": doc_name,
                        "page_number": page_idx,
                        "image_file": os.path.basename(image_path),
                        "error": str(e),
                    }
                )

    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)

        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]

        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")

        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")

        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")

In [14]:
# Run extraction for ChatGPT 5.4
try:
    process_pdf_folders(ask_about_images_json_chatgpt, ROOT_PDF_FOLDER, model_suffix="chatgpt")
except Exception as e:
    print(f"✗ Error during processing: {e}")



Starting OCR extraction for ChatGPT 5.5
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt55_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[12/28] P

# kimi_k2.5

In [54]:
# Load Kimi credentials
from dotenv import load_dotenv
from openai import OpenAI
# Load credentials from .env file
load_dotenv(override=True)
kimi_endpoint = os.getenv("kimi_endpoint")
kimi_deployment_name = os.getenv("kimi_deployment_name")
kimi_api_key = os.getenv("kimi_api_key")

# Create OpenAI-compatible client like the working kimi_k25 notebook pattern.
kimi_client = OpenAI(base_url=kimi_endpoint, api_key=kimi_api_key)

print("Kimi Credentials Loaded:")
print(f"✓ Endpoint: {kimi_endpoint}")
print(f"✓ Deployment: {kimi_deployment_name}")
print(f"✓ API Key: {kimi_api_key[:20]}..." if kimi_api_key else "✗ API Key: NOT FOUND")


Kimi Credentials Loaded:
✓ Endpoint: https://sagility-extraction-resource.openai.azure.com/openai/v1
✓ Deployment: Kimi-K2.5
✓ API Key: IJFLE9tdZO8LUekTV5Kj...


In [55]:
def ask_about_images_json_kimi(image_paths: list[str], prompt: str) -> str:
    """
    Send images to Kimi endpoint and get JSON response text.
    Uses staged token escalation to handle Kimi's reasoning token consumption.
    """
    if not all([kimi_endpoint, kimi_deployment_name, kimi_api_key]):
        raise ValueError("Missing Kimi credentials (endpoint/deployment/api_key)")

    content = []

    for image_path in image_paths:
        with open(image_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("utf-8")

        ext = image_path.split(".")[-1].lower()
        mime = {
            "jpg": "image/jpeg",
            "jpeg": "image/jpeg",
            "png": "image/png",
            "webp": "image/webp",
        }.get(ext, "image/png")

        content.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:{mime};base64,{b64}", "detail": "low"},
            }
        )

    def extract_message_text(message_content: Any) -> str:
        if isinstance(message_content, str):
            return message_content.strip()
        if isinstance(message_content, list):
            text_parts = []
            for part in message_content:
                if isinstance(part, dict) and part.get("type") == "text" and part.get("text"):
                    text_parts.append(str(part["text"]).strip())
            return "\n".join(p for p in text_parts if p).strip()
        return str(message_content).strip()

    content.append({"type": "text", "text": prompt})

    compact_prompt = (
        "Extract healthcare claim fields from this one page and return ONLY one JSON object "
        "with these keys exactly: " + ", ".join(EXTRACTION_SCHEMA.keys()) + ". "
        "Use empty string or empty array for missing values."
    )
    compact_content = content[:-1] + [{"type": "text", "text": compact_prompt}]

    # Staged attempts: escalate tokens on length-stop empty responses.
    attempts = [
        {"messages": [{"role": "user", "content": content}],       "max_tokens": 8000,  "response_format": {"type": "json_object"}},
        {"messages": [{"role": "user", "content": content}],       "max_tokens": 16000},
        {"messages": [{"role": "user", "content": compact_content}], "max_tokens": 16000},
    ]

    last_finish_reason = "unknown"
    for attempt in attempts:
        try:
            kwargs = {
                "model": kimi_deployment_name,
                "temperature": 0,
                **attempt,
            }
            completion = kimi_client.chat.completions.create(**kwargs)
        except Exception:
            kwargs_no_fmt = {k: v for k, v in kwargs.items() if k != "response_format"}
            completion = kimi_client.chat.completions.create(**kwargs_no_fmt)

        last_finish_reason = str(completion.choices[0].finish_reason)
        text = extract_message_text(completion.choices[0].message.content)
        if text:
            return text
        # If finish_reason is not length, no point escalating tokens
        if last_finish_reason != "length":
            break

    raise ValueError(
        f"Empty model response after all attempts (last_finish_reason={last_finish_reason})."
    )

In [56]:
def process_documents_to_excel_kimi(images_folder: str, output_excel: str) -> None:
    """
    Main orchestration function to process all documents with Kimi.

    Args:
        images_folder: Path to folder containing document subfolders with images
        output_excel: Path to save results Excel file
    """
    print(f"\n{'='*60}")
    print("Starting OCR extraction for Kimi")
    print(f"Images folder: {images_folder}")
    print(f"Output file: {output_excel}")
    print(f"{'='*60}\n")

    images_by_document = get_images_from_folder(images_folder)
    print(f"Found {len(images_by_document)} documents with images\n")

    all_results = []
    failed_pages = []

    # Process each document
    for doc_idx, (doc_name, image_paths) in enumerate(sorted(images_by_document.items()), 1):
        print(f"[{doc_idx}/{len(images_by_document)}] Processing: {doc_name}")

        # Process each page
        for page_idx, image_path in enumerate(image_paths, 1):
            try:
                # Get model response
                raw_response = ask_about_images_json_kimi([image_path], EXTRACTION_PROMPT)

                # Try to parse JSON
                try:
                    extracted_data = parse_model_json(raw_response)
                    parse_status = "json_ok"
                except Exception:
                    # Fallback to regex extraction
                    extracted_data = extract_partial_fields(raw_response)
                    parse_status = "partial_from_raw" if any(extracted_data.values()) else "json_failed"

                # Build result row
                page_doc_name = build_page_document_name(doc_name, page_idx)
                result = {
                    "document_name": page_doc_name,
                    "page_number": page_idx,
                    "parse_status": parse_status,
                    "extracted_data": extracted_data,
                }

                # Flatten and add to results
                flat_row = flatten_for_excel(result)
                all_results.append(flat_row)

                print(f"  Page {page_idx}: {parse_status} ✓")

            except Exception as e:
                print(f"  Page {page_idx}: ERROR - {str(e)[:50]}")
                failed_pages.append(
                    {
                        "document": doc_name,
                        "page_number": page_idx,
                        "image_file": os.path.basename(image_path),
                        "error": str(e),
                    }
                )

    # Create DataFrame and save Excel
    if all_results:
        df = pd.DataFrame(all_results)

        # Reorder columns
        column_order = ["document_name", "page_number", "parse_status"] + SCALAR_FIELDS + list(ARRAY_FIELDS)
        existing_cols = [col for col in column_order if col in df.columns]
        df = df[existing_cols]

        # Save Excel
        df.to_excel(output_excel, index=False, engine="openpyxl")
        print(f"\n✓ Saved {len(df)} results to: {output_excel}\n")

        # Save failure log
        if failed_pages:
            failure_log_path = output_excel.replace(".xlsx", "_json_failures.json")
            with open(failure_log_path, "w") as f:
                json.dump(failed_pages, f, indent=2)
            print(f"✓ Saved {len(failed_pages)} failures to: {failure_log_path}\n")

        # Preview
        print("Preview of results:")
        print(df[["document_name", "page_number", "parse_status"]].head(10).to_string())
    else:
        print("✗ No results to save")

In [57]:
# Configure paths
IMAGES_FOLDER_KIMI = r"C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images"
OUTPUT_EXCEL_KIMI = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_kimi_doc5.xlsx"


# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_EXCEL_KIMI), exist_ok=True)

# Run extraction
try:
    process_documents_to_excel_kimi(IMAGES_FOLDER_KIMI, OUTPUT_EXCEL_KIMI)
except Exception as e:
    print(f"✗ Error during processing: {e}")
    import traceback
    traceback.print_exc()


Starting OCR extraction for Kimi
Images folder: C:\Users\Rohit.Sahu\Desktop\experiment\pdf_images\val_images
Output file: C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_kimi_doc5.xlsx

Found 28 documents with images

[1/28] Processing: 220260421FICIWCA75464_page2
  Page 1: json_ok ✓
[2/28] Processing: 220260421FICIWCA75464_page4
  Page 1: json_ok ✓
[3/28] Processing: 220260421FICIWCA75465_page2
  Page 1: json_ok ✓
[4/28] Processing: 220260421FICIWCA75467_page2
  Page 1: json_ok ✓
[5/28] Processing: 220260421FICIWCA75473_page2
  Page 1: json_ok ✓
[6/28] Processing: 220260421FICIWCA75491_page2
  Page 1: json_ok ✓
[7/28] Processing: 220260421FICIWCA75495_page2
  Page 1: json_ok ✓
[8/28] Processing: 220260421FICIWCA75500_page2
  Page 1: json_ok ✓
[9/28] Processing: 220260421FICIWCA75502_page2
  Page 1: json_ok ✓
[10/28] Processing: 220260421FICIWCA75507_page2
  Page 1: json_ok ✓
[11/28] Processing: 220260421FICIWCA75507_page4
  Page 1: json_ok ✓
[12/28] Processing: 2

# comparison with chatgpt5.4 with all the models data 
# Claude and llama comparison as well


In [48]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from typing import Dict, List, Tuple, Any
import re
from collections import defaultdict

# For fuzzy matching
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


In [49]:
# Define paths
experiment_dir = Path(r"c:\Users\Rohit.Sahu\Desktop\experiment")
extraction_results_dir = experiment_dir / "output"

# Field definitions for reference
ARRAY_FIELDS = {"diagnosis_codes", "cpt_codes", "charges", "units"}
SCALAR_FIELDS = [
    "claimant_name",
    "claimant_number",
    "tax_id",
    "practice_address",
    "billing_address",
    "date_of_service",
    "invoice_date",
    "invoice_number",
    "taxonomy",
    "total_amount",
]

print(f"Experiment directory: {experiment_dir}")
print(f"Results directory: {extraction_results_dir}")
print("\nField definitions loaded:")
print(f"  Array fields: {ARRAY_FIELDS}")
print(f"  Scalar fields: {len(SCALAR_FIELDS)}")
print(f"  Scalar fields: {len(SCALAR_FIELDS)} fields")

Experiment directory: c:\Users\Rohit.Sahu\Desktop\experiment
Results directory: c:\Users\Rohit.Sahu\Desktop\experiment\output

Field definitions loaded:
  Array fields: {'charges', 'units', 'cpt_codes', 'diagnosis_codes'}
  Scalar fields: 10
  Scalar fields: 10 fields


In [50]:
# Load GPT results
gpt_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt54_doc5.xlsx"
llama_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_llama_maverick_doc5.xlsx"
llama_scout_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_llama_scout_doc5.xlsx"
kimi_k25_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_kimi_doc5.xlsx"
claude_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_claude_doc5.xlsx"
gpt_nano_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt54_nano_doc5.xlsx"
# Load Amazon Nova Lite and Qwen results
nova_lite_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_nova_lite_doc5.xlsx"
qwen_excel = r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_qwen3_235b_a22b.xlsx"
gpt_55=r"C:\Users\Rohit.Sahu\Desktop\experiment\output\OCR_Extraction_Results_chatgpt55_doc5.xlsx"
try:
    gpt_df = pd.read_excel(gpt_excel)
    print(f"✓ Loaded GPT results: {len(gpt_df)} records, {len(gpt_df.columns)} columns")
    print(f"  Columns: {list(gpt_df.columns)}")
except FileNotFoundError:
    print(f"✗ GPT file not found: {gpt_excel}")
    gpt_df = None

try:
    llama_df = pd.read_excel(llama_excel)
    print(f"✓ Loaded Llama results: {len(llama_df)} records, {len(llama_df.columns)} columns")
    print(f"  Columns: {list(llama_df.columns)}")
except FileNotFoundError:
    print(f"✗ Llama file not found: {llama_excel}")
    llama_df = None

try:
    llama_scout_df = pd.read_excel(llama_scout_excel)
    print(f"✓ Loaded Llama Scout results: {len(llama_scout_df)} records, {len(llama_scout_df.columns)} columns")
    print(f"  Columns: {list(llama_scout_df.columns)}")
except FileNotFoundError:
    print(f"✗ Llama Scout file not found: {llama_scout_excel}")
    llama_scout_df = None

try:
    kimi_k25_df = pd.read_excel(kimi_k25_excel)
    print(f"✓ Loaded Kimi-K2.5 results: {len(kimi_k25_df)} records, {len(kimi_k25_df.columns)} columns")
    print(f"  Columns: {list(kimi_k25_df.columns)}")
except FileNotFoundError:
    print(f"⚠ Kimi-K2.5 file not found: {kimi_k25_excel}")
    kimi_k25_df = None

try:
    claude_df = pd.read_excel(claude_excel)
    print(f"✓ Loaded Claude results: {len(claude_df)} records, {len(claude_df.columns)} columns")
    print(f"  Columns: {list(claude_df.columns)}")
except FileNotFoundError:
    print(f"⚠ Claude file not found: {claude_excel}")
    claude_df = None

try:
    gpt_nano_df = pd.read_excel(gpt_nano_excel)
    print(f"✓ Loaded GPT Nano results: {len(gpt_nano_df)} records, {len(gpt_nano_df.columns)} columns")
    print(f"  Columns: {list(gpt_nano_df.columns)}")
except FileNotFoundError:
    print(f"⚠ GPT Nano file not found: {gpt_nano_excel}")
    gpt_nano_df = None

try:
    gpt_55_df = pd.read_excel(gpt_55)
    print(f"✓ Loaded GPT 5.5 results: {len(gpt_55_df)} records, {len(gpt_55_df.columns)} columns")
    print(f"  Columns: {list(gpt_55_df.columns)}")
except FileNotFoundError:
    print(f"✗ GPT file not found: {gpt_excel}")
    gpt_55_df = None

try:
    nova_lite_df = pd.read_excel(nova_lite_excel)
    print(f"✓ Loaded Amazon Nova Lite results: {len(nova_lite_df)} records, {len(nova_lite_df.columns)} columns")
    print(f"  Columns: {list(nova_lite_df.columns)}")
except FileNotFoundError:
    print(f"⚠ Amazon Nova Lite file not found: {nova_lite_excel}")
    nova_lite_df = None

try:
    qwen_df = pd.read_excel(qwen_excel)
    print(f"✓ Loaded Qwen 23.5B results: {len(qwen_df)} records, {len(qwen_df.columns)} columns")
    print(f"  Columns: {list(qwen_df.columns)}")
except FileNotFoundError:
    print(f"⚠ Qwen file not found: {qwen_excel}")
    qwen_df = None

# Display first few rows
if gpt_df is not None:
    print("\nGPT Results Preview:")
    print(gpt_df.head(2))


def normalize_for_exact_match(value: Any) -> str:
    """
    Normalize values for exact matching by:
    - Converting to string
    - Trimming whitespace
    - Converting to lowercase
    - Removing extra spaces
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""

    normalized = str(value).strip()
    normalized = normalized.lower()
    normalized = re.sub(r"\s+", " ", normalized)
    return normalized


def exact_match_scalar(expected: Any, extracted: Any) -> bool:
    """
    Perform exact character-by-character match after normalization.
    Returns 1 (match) or 0 (no match).
    """
    exp_norm = normalize_for_exact_match(expected)
    ext_norm = normalize_for_exact_match(extracted)
    return 1 if exp_norm == ext_norm else 0


def exact_match_array(expected: List[Any], extracted: List[Any]) -> bool:
    """
    Perform exact match for array fields by:
    - Normalizing each element
    - Sorting both lists
    - Comparing element by element

    Returns 1 if all elements match exactly (in any order), 0 otherwise.
    """
    if not isinstance(expected, list):
        expected = [expected] if expected else []
    if not isinstance(extracted, list):
        extracted = [extracted] if extracted else []

    exp_norm = sorted([normalize_for_exact_match(x) for x in expected if x])
    ext_norm = sorted([normalize_for_exact_match(x) for x in extracted if x])
    return 1 if exp_norm == ext_norm else 0


print("✓ Exact match functions defined")

✓ Loaded GPT results: 28 records, 21 columns
  Columns: ['document_name', 'page_number', 'parse_status', 'inference_ms', 'input_tokens', 'output_tokens', 'total_tokens', 'claimant_name', 'claimant_number', 'tax_id', 'practice_address', 'billing_address', 'date_of_service', 'invoice_date', 'invoice_number', 'taxonomy', 'total_amount', 'charges', 'units', 'cpt_codes', 'diagnosis_codes']
✓ Loaded Llama results: 28 records, 21 columns
  Columns: ['document_name', 'page_number', 'parse_status', 'inference_ms', 'input_tokens', 'output_tokens', 'total_tokens', 'claimant_name', 'claimant_number', 'tax_id', 'practice_address', 'billing_address', 'date_of_service', 'invoice_date', 'invoice_number', 'taxonomy', 'total_amount', 'charges', 'units', 'cpt_codes', 'diagnosis_codes']
✓ Loaded Llama Scout results: 28 records, 21 columns
  Columns: ['document_name', 'page_number', 'parse_status', 'inference_ms', 'input_tokens', 'output_tokens', 'total_tokens', 'claimant_name', 'claimant_number', 'tax_id'

In [51]:
# Field-level fuzzy strategy and threshold configuration
FIELD_FUZZY_STRATEGY = {
    "claimant_number": "ratio",
    "tax_id": "ratio",
    "invoice_number": "ratio",
    "date_of_service": "ratio",
    "invoice_date": "ratio",
    "Pharmacy_Name": "token_sort",
    "claimant_name": "token_sort",
    "practice_address": "token_sort",
    "billing_address": "token_sort",
}

# Per-field thresholds tuned for OCR extraction quality
FIELD_FUZZY_THRESHOLD = {
    "claimant_number": 95,
    "tax_id": 95,
    "invoice_number": 95,
    "date_of_service": 95,
    "invoice_date": 95,
    "taxonomy": 95,
    "total_amount": 95,
    "claimant_name": 88,
    "practice_address": 85,
    "billing_address": 85,
}

DEFAULT_FUZZY_THRESHOLD = 90


def get_fuzzy_scorer(strategy: str):
    """Return fuzzywuzzy scorer function for a strategy key."""
    strategy_map = {
        "ratio": fuzz.ratio,
        "token_sort": fuzz.token_sort_ratio,
        "token_set": fuzz.token_set_ratio,
        "partial": fuzz.partial_ratio,
    }
    return strategy_map.get(strategy, fuzz.token_sort_ratio)


def normalize_for_fuzzy_field(value: Any, field_name: str = "") -> str:
    """Field-aware normalization before fuzzy scoring."""
    text = normalize_for_exact_match(value)
    if not text:
        return ""

    # IDs and code-like fields: keep alphanumeric only for stable OCR comparison.
    if field_name in {"claimant_number", "invoice_number", "taxonomy"}:
        return re.sub(r"[^a-z0-9]", "", text)

    # Tax ID: compare only digits to ignore formatting differences like XX-XXXXXXX.
    if field_name == "tax_id":
        return re.sub(r"\D", "", text)

    # Dates: normalize separators and remove spaces.
    if field_name in {"invoice_date", "date_of_service"}:
        text = re.sub(r"[-.]", "/", text)
        return re.sub(r"\s+", "", text)

    # Currency / totals: keep numeric core (digits + decimal point).
    if field_name in {"total_amount"}:
        text = text.replace(",", "")
        text = text.replace("$", "")
        return re.sub(r"[^0-9.]", "", text)

    # Default for names/addresses/free text.
    return text


def fuzzy_match_scalar(
    expected: Any,
    extracted: Any,
    field_name: str = "",
    threshold: int = DEFAULT_FUZZY_THRESHOLD,
) -> Tuple[int, int]:
    """
    Perform fuzzy matching for scalar fields with field-specific strategy.

    Args:
        expected: Expected value
        extracted: Extracted value
        field_name: Field name used to select strategy and normalization
        threshold: Similarity threshold (0-100) when field override is not defined

    Returns:
        Tuple of (match_result: 0 or 1, similarity_score: 0-100)
    """
    exp_norm = normalize_for_fuzzy_field(expected, field_name)
    ext_norm = normalize_for_fuzzy_field(extracted, field_name)

    # Handle empty cases
    if not exp_norm and not ext_norm:
        return (1, 100)
    if not exp_norm or not ext_norm:
        return (0, 0)

    strategy = FIELD_FUZZY_STRATEGY.get(field_name, "token_sort")
    scorer = get_fuzzy_scorer(strategy)
    score = scorer(exp_norm, ext_norm)

    field_threshold = FIELD_FUZZY_THRESHOLD.get(field_name, threshold)
    match = 1 if score >= field_threshold else 0
    return (match, score)


def fuzzy_match_array(expected: List[Any], extracted: List[Any], threshold: int = DEFAULT_FUZZY_THRESHOLD) -> Tuple[int, float]:
    """
    Perform fuzzy matching for array fields (codes, charges, etc).
    Matches each expected element with extracted elements using fuzzy matching.

    Args:
        expected: Expected values list
        extracted: Extracted values list
        threshold: Similarity threshold for each item

    Returns:
        Tuple of (match_result: 0 or 1, match_ratio: 0.0-1.0 for proportion matched)
    """
    if not isinstance(expected, list):
        expected = [expected] if expected else []
    if not isinstance(extracted, list):
        extracted = [extracted] if extracted else []

    # Normalize all values and drop empties after normalization
    exp_norm = [val for x in expected if (val := normalize_for_exact_match(x))]
    ext_norm = [val for x in extracted if (val := normalize_for_exact_match(x))]

    # Handle empty cases
    if not exp_norm and not ext_norm:
        return (1, 1.0)
    if not exp_norm or not ext_norm:
        return (0, 0.0)

    # Try to match each expected item with best option in extracted
    matched_count = 0
    for exp_item in exp_norm:
        best_match = process.extractOne(exp_item, ext_norm, scorer=fuzz.token_sort_ratio)
        if best_match and best_match[1] >= threshold:
            matched_count += 1

    match_ratio = matched_count / len(exp_norm) if exp_norm else 0.0

    # Consider it a match if at least 80% of items matched
    match = 1 if match_ratio >= 0.8 else 0

    return (match, match_ratio)


print(f"✓ Fuzzy match functions defined (default threshold={DEFAULT_FUZZY_THRESHOLD}%)")


✓ Fuzzy match functions defined (default threshold=90%)


In [52]:
def apply_matching_strategies(
    gpt_df: pd.DataFrame,
    llama_df: pd.DataFrame,
    llama_scout_df: pd.DataFrame,
    gpt_nano_df: pd.DataFrame = None,
    gpt_55_df: pd.DataFrame = None,
    kimi_k25_df: pd.DataFrame = None,
    claude_df: pd.DataFrame = None,
    nova_lite_df: pd.DataFrame = None,
    qwen_df: pd.DataFrame = None,
    threshold: int = DEFAULT_FUZZY_THRESHOLD,
) -> Dict[str, pd.DataFrame]:
    """
    Apply exact and fuzzy matching for:
    1) GPT baseline comparisons (model vs GPT)
    2) Claude baseline comparisons (model vs Claude), when Claude data is available
    """
    if gpt_df is None or llama_df is None or llama_scout_df is None:
        print("✗ Cannot compare - missing required dataframes (gpt/llama/scout)")
        return {}

    # Record count validation for GPT baseline
    if len(gpt_df) != len(llama_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Llama ({len(llama_df)})")
    if len(gpt_df) != len(llama_scout_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Scout ({len(llama_scout_df)})")
    if gpt_nano_df is not None and len(gpt_df) != len(gpt_nano_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs GPT Nano ({len(gpt_nano_df)})")
    if gpt_55_df is not None and len(gpt_df) != len(gpt_55_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs GPT Nano ({len(gpt_55_df)})")
    if kimi_k25_df is not None and len(gpt_df) != len(kimi_k25_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Kimi-K2.5 ({len(kimi_k25_df)})")
    if claude_df is not None and len(gpt_df) != len(claude_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Claude ({len(claude_df)})")
    if nova_lite_df is not None and len(gpt_df) != len(nova_lite_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Nova Lite ({len(nova_lite_df)})")
    if qwen_df is not None and len(gpt_df) != len(qwen_df):
        print(f"⚠ Warning: GPT ({len(gpt_df)}) vs Qwen ({len(qwen_df)})")

    # Minimum rows for GPT baseline comparisons
    min_rows_llama = min(len(gpt_df), len(llama_df))
    min_rows_scout = min(len(gpt_df), len(llama_scout_df))
    min_rows_nano = min(len(gpt_df), len(gpt_nano_df)) if gpt_nano_df is not None else 0
    min_rows_55 = min(len(gpt_df), len(gpt_55_df)) if gpt_55_df is not None else 0
    min_rows_kimi = min(len(gpt_df), len(kimi_k25_df)) if kimi_k25_df is not None else 0
    min_rows_claude = min(len(gpt_df), len(claude_df)) if claude_df is not None else 0
    min_rows_nova = min(len(gpt_df), len(nova_lite_df)) if nova_lite_df is not None else 0
    min_rows_qwen = min(len(gpt_df), len(qwen_df)) if qwen_df is not None else 0

    # Result holders for GPT baseline
    llama_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    scout_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    nano_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    gpt_55_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    kimi_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    nova_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    qwen_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}

    # Result holders for Claude baseline
    claude_vs_llama_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_scout_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_nano_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_gpt_55_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_kimi_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_nova_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}
    claude_vs_qwen_results = {"exact_match": [], "fuzzy_match": [], "detailed": []}

    def _compare_rows(base_row, model_row, threshold):
        exact_res = {"record_idx": base_row.name}
        fuzzy_res = {"record_idx": base_row.name}
        detailed = {"record_idx": base_row.name}

        for field in SCALAR_FIELDS:
            if field in base_row.index:
                exact_match_result = exact_match_scalar(base_row[field], model_row[field])
                fuzzy_match_result, score = fuzzy_match_scalar(
                    base_row[field], model_row[field], field_name=field, threshold=threshold
                )
                exact_res[f"{field}_exact"] = exact_match_result
                fuzzy_res[f"{field}_fuzzy"] = fuzzy_match_result
                detailed[f"{field}_score"] = score

        for field in ARRAY_FIELDS:
            if field in base_row.index:
                exact_match_result = exact_match_array(base_row[field], model_row[field])
                fuzzy_match_result, ratio = fuzzy_match_array(
                    base_row[field], model_row[field], threshold=threshold
                )
                exact_res[f"{field}_exact"] = exact_match_result
                fuzzy_res[f"{field}_fuzzy"] = fuzzy_match_result
                detailed[f"{field}_ratio"] = ratio

        return exact_res, fuzzy_res, detailed

    # GPT baseline comparisons
    print("Running comparisons with GPT baseline...")

    for idx in range(min_rows_llama):
        e, f, d = _compare_rows(gpt_df.iloc[idx], llama_df.iloc[idx], threshold)
        llama_results["exact_match"].append(e)
        llama_results["fuzzy_match"].append(f)
        llama_results["detailed"].append(d)

    for idx in range(min_rows_scout):
        e, f, d = _compare_rows(gpt_df.iloc[idx], llama_scout_df.iloc[idx], threshold)
        scout_results["exact_match"].append(e)
        scout_results["fuzzy_match"].append(f)
        scout_results["detailed"].append(d)

    if gpt_nano_df is not None:
        for idx in range(min_rows_nano):
            e, f, d = _compare_rows(gpt_df.iloc[idx], gpt_nano_df.iloc[idx], threshold)
            nano_results["exact_match"].append(e)
            nano_results["fuzzy_match"].append(f)
            nano_results["detailed"].append(d)
    
    if gpt_55_df is not None:
        for idx in range(min_rows_55):
            e, f, d = _compare_rows(gpt_df.iloc[idx], gpt_55_df.iloc[idx], threshold)
            gpt_55_results["exact_match"].append(e)
            gpt_55_results["fuzzy_match"].append(f)
            gpt_55_results["detailed"].append(d)

    if kimi_k25_df is not None:
        for idx in range(min_rows_kimi):
            e, f, d = _compare_rows(gpt_df.iloc[idx], kimi_k25_df.iloc[idx], threshold)
            kimi_results["exact_match"].append(e)
            kimi_results["fuzzy_match"].append(f)
            kimi_results["detailed"].append(d)

    if claude_df is not None:
        for idx in range(min_rows_claude):
            e, f, d = _compare_rows(gpt_df.iloc[idx], claude_df.iloc[idx], threshold)
            claude_results["exact_match"].append(e)
            claude_results["fuzzy_match"].append(f)
            claude_results["detailed"].append(d)

    if nova_lite_df is not None:
        for idx in range(min_rows_nova):
            e, f, d = _compare_rows(gpt_df.iloc[idx], nova_lite_df.iloc[idx], threshold)
            nova_results["exact_match"].append(e)
            nova_results["fuzzy_match"].append(f)
            nova_results["detailed"].append(d)

    if qwen_df is not None:
        for idx in range(min_rows_qwen):
            e, f, d = _compare_rows(gpt_df.iloc[idx], qwen_df.iloc[idx], threshold)
            qwen_results["exact_match"].append(e)
            qwen_results["fuzzy_match"].append(f)
            qwen_results["detailed"].append(d)

    # Claude baseline comparisons
    if claude_df is not None:
        print("Running comparisons with Claude baseline...")

        min_rows_claude_vs_llama = min(len(claude_df), len(llama_df))
        min_rows_claude_vs_scout = min(len(claude_df), len(llama_scout_df))
        min_rows_claude_vs_nano = min(len(claude_df), len(gpt_nano_df)) if gpt_nano_df is not None else 0
        min_rows_claude_vs_gpt_55 = min(len(claude_df), len(gpt_55_df)) if gpt_55_df is not None else 0
        min_rows_claude_vs_kimi = min(len(claude_df), len(kimi_k25_df)) if kimi_k25_df is not None else 0
        min_rows_claude_vs_nova = min(len(claude_df), len(nova_lite_df)) if nova_lite_df is not None else 0
        min_rows_claude_vs_qwen = min(len(claude_df), len(qwen_df)) if qwen_df is not None else 0

        for idx in range(min_rows_claude_vs_llama):
            e, f, d = _compare_rows(claude_df.iloc[idx], llama_df.iloc[idx], threshold)
            claude_vs_llama_results["exact_match"].append(e)
            claude_vs_llama_results["fuzzy_match"].append(f)
            claude_vs_llama_results["detailed"].append(d)

        for idx in range(min_rows_claude_vs_scout):
            e, f, d = _compare_rows(claude_df.iloc[idx], llama_scout_df.iloc[idx], threshold)
            claude_vs_scout_results["exact_match"].append(e)
            claude_vs_scout_results["fuzzy_match"].append(f)
            claude_vs_scout_results["detailed"].append(d)

        if gpt_nano_df is not None:
            for idx in range(min_rows_claude_vs_nano):
                e, f, d = _compare_rows(claude_df.iloc[idx], gpt_nano_df.iloc[idx], threshold)
                claude_vs_nano_results["exact_match"].append(e)
                claude_vs_nano_results["fuzzy_match"].append(f)
                claude_vs_nano_results["detailed"].append(d)
        
        if gpt_55_df is not None:
            for idx in range(min_rows_claude_vs_gpt_55):
                e, f, d = _compare_rows(claude_df.iloc[idx], gpt_55_df.iloc[idx], threshold)
                claude_vs_gpt_55_results["exact_match"].append(e)
                claude_vs_gpt_55_results["fuzzy_match"].append(f)
                claude_vs_gpt_55_results["detailed"].append(d)
                
        if kimi_k25_df is not None:
            for idx in range(min_rows_claude_vs_kimi):
                e, f, d = _compare_rows(claude_df.iloc[idx], kimi_k25_df.iloc[idx], threshold)
                claude_vs_kimi_results["exact_match"].append(e)
                claude_vs_kimi_results["fuzzy_match"].append(f)
                claude_vs_kimi_results["detailed"].append(d)

        if nova_lite_df is not None:
            for idx in range(min_rows_claude_vs_nova):
                e, f, d = _compare_rows(claude_df.iloc[idx], nova_lite_df.iloc[idx], threshold)
                claude_vs_nova_results["exact_match"].append(e)
                claude_vs_nova_results["fuzzy_match"].append(f)
                claude_vs_nova_results["detailed"].append(d)

        if qwen_df is not None:
            for idx in range(min_rows_claude_vs_qwen):
                e, f, d = _compare_rows(claude_df.iloc[idx], qwen_df.iloc[idx], threshold)
                claude_vs_qwen_results["exact_match"].append(e)
                claude_vs_qwen_results["fuzzy_match"].append(f)
                claude_vs_qwen_results["detailed"].append(d)

    # Build result dictionary
    result = {
        "exact_match": pd.DataFrame(llama_results["exact_match"]),
        "fuzzy_match": pd.DataFrame(llama_results["fuzzy_match"]),
        "detailed": pd.DataFrame(llama_results["detailed"]),
        "exact_match_scout": pd.DataFrame(scout_results["exact_match"]),
        "fuzzy_match_scout": pd.DataFrame(scout_results["fuzzy_match"]),
        "detailed_scout": pd.DataFrame(scout_results["detailed"]),
    }

    if gpt_nano_df is not None:
        result.update(
            {
                "exact_match_nano": pd.DataFrame(nano_results["exact_match"]),
                "fuzzy_match_nano": pd.DataFrame(nano_results["fuzzy_match"]),
                "detailed_nano": pd.DataFrame(nano_results["detailed"]),
            }
        )

    if gpt_55_df is not None:
        result.update(
        {
            "exact_match_gpt_55": pd.DataFrame(gpt_55_results["exact_match"]),
            "fuzzy_match_gpt_55": pd.DataFrame(gpt_55_results["fuzzy_match"]),
            "detailed_gpt_55": pd.DataFrame(gpt_55_results["detailed"]),
        }
    )

    if kimi_k25_df is not None:
        result.update(
            {
                "exact_match_kimi": pd.DataFrame(kimi_results["exact_match"]),
                "fuzzy_match_kimi": pd.DataFrame(kimi_results["fuzzy_match"]),
                "detailed_kimi": pd.DataFrame(kimi_results["detailed"]),
            }
        )

    if claude_df is not None:
        result.update(
            {
                "exact_match_claude": pd.DataFrame(claude_results["exact_match"]),
                "fuzzy_match_claude": pd.DataFrame(claude_results["fuzzy_match"]),
                "detailed_claude": pd.DataFrame(claude_results["detailed"]),
                "exact_match_claude_vs_llama": pd.DataFrame(claude_vs_llama_results["exact_match"]),
                "fuzzy_match_claude_vs_llama": pd.DataFrame(claude_vs_llama_results["fuzzy_match"]),
                "detailed_claude_vs_llama": pd.DataFrame(claude_vs_llama_results["detailed"]),
                "exact_match_claude_vs_scout": pd.DataFrame(claude_vs_scout_results["exact_match"]),
                "fuzzy_match_claude_vs_scout": pd.DataFrame(claude_vs_scout_results["fuzzy_match"]),
                "detailed_claude_vs_scout": pd.DataFrame(claude_vs_scout_results["detailed"]),
            }
        )

        if gpt_nano_df is not None:
            result.update(
                {
                    "exact_match_claude_vs_nano": pd.DataFrame(claude_vs_nano_results["exact_match"]),
                    "fuzzy_match_claude_vs_nano": pd.DataFrame(claude_vs_nano_results["fuzzy_match"]),
                    "detailed_claude_vs_nano": pd.DataFrame(claude_vs_nano_results["detailed"]),
                }
            )
        
        if gpt_55_df is not None:
            result.update(
        {
            "exact_match_claude_vs_gpt_55": pd.DataFrame(claude_vs_gpt_55_results["exact_match"]),
            "fuzzy_match_claude_vs_gpt_55": pd.DataFrame(claude_vs_gpt_55_results["fuzzy_match"]),
            "detailed_claude_vs_gpt_55": pd.DataFrame(claude_vs_gpt_55_results["detailed"]),
        }
    )

        if kimi_k25_df is not None:
            result.update(
                {
                    "exact_match_claude_vs_kimi": pd.DataFrame(claude_vs_kimi_results["exact_match"]),
                    "fuzzy_match_claude_vs_kimi": pd.DataFrame(claude_vs_kimi_results["fuzzy_match"]),
                    "detailed_claude_vs_kimi": pd.DataFrame(claude_vs_kimi_results["detailed"]),
                }
            )

        if nova_lite_df is not None:
            result.update(
                {
                    "exact_match_claude_vs_nova": pd.DataFrame(claude_vs_nova_results["exact_match"]),
                    "fuzzy_match_claude_vs_nova": pd.DataFrame(claude_vs_nova_results["fuzzy_match"]),
                    "detailed_claude_vs_nova": pd.DataFrame(claude_vs_nova_results["detailed"]),
                }
            )

        if qwen_df is not None:
            result.update(
                {
                    "exact_match_claude_vs_qwen": pd.DataFrame(claude_vs_qwen_results["exact_match"]),
                    "fuzzy_match_claude_vs_qwen": pd.DataFrame(claude_vs_qwen_results["fuzzy_match"]),
                    "detailed_claude_vs_qwen": pd.DataFrame(claude_vs_qwen_results["detailed"]),
                }
            )

    if nova_lite_df is not None:
        result.update(
            {
                "exact_match_nova": pd.DataFrame(nova_results["exact_match"]),
                "fuzzy_match_nova": pd.DataFrame(nova_results["fuzzy_match"]),
                "detailed_nova": pd.DataFrame(nova_results["detailed"]),
            }
        )

    if qwen_df is not None:
        result.update(
            {
                "exact_match_qwen": pd.DataFrame(qwen_results["exact_match"]),
                "fuzzy_match_qwen": pd.DataFrame(qwen_results["fuzzy_match"]),
                "detailed_qwen": pd.DataFrame(qwen_results["detailed"]),
            }
        )

    return result


# Run comparison including Amazon Nova Lite and Qwen
if gpt_df is not None and llama_df is not None and llama_scout_df is not None:
    print("Running comparison with all models...")
    comparison_results = apply_matching_strategies(
    gpt_df=gpt_df,
    llama_df=llama_df,
    llama_scout_df=llama_scout_df,
    gpt_nano_df=gpt_nano_df,
    gpt_55_df=gpt_55_df,
    kimi_k25_df=kimi_k25_df,
    claude_df=claude_df,
    nova_lite_df=nova_lite_df,
    qwen_df=qwen_df,
    threshold=DEFAULT_FUZZY_THRESHOLD,
)
    print(f"✓ Llama comparison complete: {len(comparison_results['exact_match'])} records processed")
    print(f"✓ Scout comparison complete: {len(comparison_results['exact_match_scout'])} records processed")
    if "exact_match_nano" in comparison_results:
        print(f"✓ GPT Nano comparison complete: {len(comparison_results['exact_match_nano'])} records processed")
    if "exact_match_kimi" in comparison_results:
        print(f"✓ Kimi-K2.5 comparison complete: {len(comparison_results['exact_match_kimi'])} records processed")
    if "exact_match_claude" in comparison_results:
        print(f"✓ Claude comparison complete: {len(comparison_results['exact_match_claude'])} records processed")
    if "exact_match_nova" in comparison_results:
        print(f"✓ Amazon Nova Lite comparison complete: {len(comparison_results['exact_match_nova'])} records processed")
    if "exact_match_gpt_55" in comparison_results:
        print(f"✓ GPT 5.5 comparison complete: {len(comparison_results['exact_match_gpt_55'])} records processed")   
    if "exact_match_qwen" in comparison_results:
        print(f"✓ Qwen 23.5B comparison complete: {len(comparison_results['exact_match_qwen'])} records processed")
    if "exact_match_claude_vs_llama" in comparison_results:
        print(f"✓ Claude baseline (Llama) complete: {len(comparison_results['exact_match_claude_vs_llama'])} records processed")
    if "exact_match_claude_vs_scout" in comparison_results:
        print(f"✓ Claude baseline (Scout) complete: {len(comparison_results['exact_match_claude_vs_scout'])} records processed")
    if "exact_match_claude_vs_nano" in comparison_results:
        print(f"✓ Claude baseline (GPT Nano) complete: {len(comparison_results['exact_match_claude_vs_nano'])} records processed")
    if "exact_match_claude_vs_kimi" in comparison_results:
        print(f"✓ Claude baseline (Kimi-K2.5) complete: {len(comparison_results['exact_match_claude_vs_kimi'])} records processed")
    if "exact_match_claude_vs_nova" in comparison_results:
        print(f"✓ Claude baseline (Amazon Nova Lite) complete: {len(comparison_results['exact_match_claude_vs_nova'])} records processed")
    if "exact_match_claude_vs_qwen" in comparison_results:
        print(f"✓ Claude baseline (Qwen 23.5B) complete: {len(comparison_results['exact_match_claude_vs_qwen'])} records processed")
else:
    print("⚠ Skipping comparison - data not loaded")

Running comparison with all models...
Running comparisons with GPT baseline...
Running comparisons with Claude baseline...
✓ Llama comparison complete: 28 records processed
✓ Scout comparison complete: 28 records processed
✓ GPT Nano comparison complete: 28 records processed
✓ Kimi-K2.5 comparison complete: 28 records processed
✓ Claude comparison complete: 28 records processed
✓ Amazon Nova Lite comparison complete: 28 records processed
✓ GPT 5.5 comparison complete: 28 records processed
✓ Qwen 23.5B comparison complete: 28 records processed
✓ Claude baseline (Llama) complete: 28 records processed
✓ Claude baseline (Scout) complete: 28 records processed
✓ Claude baseline (GPT Nano) complete: 28 records processed
✓ Claude baseline (Kimi-K2.5) complete: 28 records processed
✓ Claude baseline (Amazon Nova Lite) complete: 28 records processed
✓ Claude baseline (Qwen 23.5B) complete: 28 records processed


In [53]:
def calculate_accuracy_metrics(comparison_df: pd.DataFrame, match_type: str = "exact") -> Dict[str, Any]:
    """
    Calculate comprehensive accuracy metrics for a matching approach.

    Args:
        comparison_df: DataFrame with match results for each field
        match_type: Either "exact" or "fuzzy"

    Returns:
        Dictionary with metrics per field and overall metrics
    """
    metrics = {
        "match_type": match_type,
        "total_records": len(comparison_df),
        "field_metrics": {},
        "overall": {},
    }

    # Get all match columns (excluding record_idx)
    match_columns = [col for col in comparison_df.columns if col != "record_idx"]

    field_accuracy = {}

    for col in match_columns:
        if col in comparison_df.columns:
            # Count matches (1) vs non-matches (0)
            matches = (comparison_df[col] == 1).sum()
            total = len(comparison_df)
            accuracy = (matches / total * 100) if total > 0 else 0

            field_name = col.replace(f"_{match_type}", "")
            field_accuracy[field_name] = accuracy

    # Calculate overall accuracy
    if field_accuracy:
        overall_accuracy = sum(field_accuracy.values()) / len(field_accuracy)
    else:
        overall_accuracy = 0

    metrics["field_metrics"] = field_accuracy
    metrics["overall"]["accuracy"] = overall_accuracy
    metrics["overall"]["average_field_accuracy"] = overall_accuracy

    return metrics


def build_model_wise_summary(
    exact_metrics: Dict[str, Any],
    fuzzy_metrics: Dict[str, Any],
    exact_metrics_scout: Dict[str, Any],
    fuzzy_metrics_scout: Dict[str, Any],
    exact_metrics_nano: Dict[str, Any] = None,
    fuzzy_metrics_nano: Dict[str, Any] = None,
    exact_metrics_55: Dict[str, Any] = None,      # ADD
    fuzzy_metrics_55: Dict[str, Any] = None,      # ADD
    exact_metrics_kimi: Dict[str, Any] = None,
    fuzzy_metrics_kimi: Dict[str, Any] = None,
    exact_metrics_claude: Dict[str, Any] = None,
    fuzzy_metrics_claude: Dict[str, Any] = None,
    exact_metrics_nova: Dict[str, Any] = None,
    fuzzy_metrics_nova: Dict[str, Any] = None,
    exact_metrics_qwen: Dict[str, Any] = None,
    fuzzy_metrics_qwen: Dict[str, Any] = None,
) -> pd.DataFrame:
    """Build a compact model-wise comparison summary for GPT baseline."""
    data = [
        {
            "model": "Llama",
            "exact_accuracy_%": exact_metrics["overall"]["accuracy"],
            "fuzzy_accuracy_%": fuzzy_metrics["overall"]["accuracy"],
            "fuzzy_minus_exact_%": fuzzy_metrics["overall"]["accuracy"] - exact_metrics["overall"]["accuracy"],
            "records_analyzed": exact_metrics["total_records"],
        },
        {
            "model": "Llama Scout",
            "exact_accuracy_%": exact_metrics_scout["overall"]["accuracy"],
            "fuzzy_accuracy_%": fuzzy_metrics_scout["overall"]["accuracy"],
            "fuzzy_minus_exact_%": fuzzy_metrics_scout["overall"]["accuracy"] - exact_metrics_scout["overall"]["accuracy"],
            "records_analyzed": exact_metrics_scout["total_records"],
        },
    ]

    if exact_metrics_nano is not None and fuzzy_metrics_nano is not None:
        data.append(
            {
                "model": "GPT Nano",
                "exact_accuracy_%": exact_metrics_nano["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_nano["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_nano["overall"]["accuracy"] - exact_metrics_nano["overall"]["accuracy"],
                "records_analyzed": exact_metrics_nano["total_records"],
            }
        )
    if exact_metrics_55 is not None and fuzzy_metrics_55 is not None:
        data.append(
            {
                "model": "GPT 5.5",
                "exact_accuracy_%": exact_metrics_55["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_55["overall"]["accuracy"],
                "fuzzy_minus_exact_%":fuzzy_metrics_55["overall"]["accuracy"] - exact_metrics_55["overall"]["accuracy"],
                "records_analyzed": exact_metrics_55["total_records"],
            }
        )
    if exact_metrics_kimi is not None and fuzzy_metrics_kimi is not None:
        data.append(
            {
                "model": "Kimi-K2.5",
                "exact_accuracy_%": exact_metrics_kimi["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_kimi["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_kimi["overall"]["accuracy"] - exact_metrics_kimi["overall"]["accuracy"],
                "records_analyzed": exact_metrics_kimi["total_records"],
            }
        )

    if exact_metrics_claude is not None and fuzzy_metrics_claude is not None:
        data.append(
            {
                "model": "Claude",
                "exact_accuracy_%": exact_metrics_claude["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_claude["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_claude["overall"]["accuracy"] - exact_metrics_claude["overall"]["accuracy"],
                "records_analyzed": exact_metrics_claude["total_records"],
            }
        )

    if exact_metrics_nova is not None and fuzzy_metrics_nova is not None:
        data.append(
            {
                "model": "Amazon Nova Lite",
                "exact_accuracy_%": exact_metrics_nova["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_nova["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_nova["overall"]["accuracy"] - exact_metrics_nova["overall"]["accuracy"],
                "records_analyzed": exact_metrics_nova["total_records"],
            }
        )

    if exact_metrics_qwen is not None and fuzzy_metrics_qwen is not None:
        data.append(
            {
                "model": "Qwen 23.5B",
                "exact_accuracy_%": exact_metrics_qwen["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_qwen["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_qwen["overall"]["accuracy"] - exact_metrics_qwen["overall"]["accuracy"],
                "records_analyzed": exact_metrics_qwen["total_records"],
            }
        )

    return pd.DataFrame(data)


def build_claude_baseline_summary(
    exact_metrics_llama: Dict[str, Any],
    fuzzy_metrics_llama: Dict[str, Any],
    exact_metrics_scout: Dict[str, Any],
    fuzzy_metrics_scout: Dict[str, Any],
    exact_metrics_nano: Dict[str, Any] = None,
    fuzzy_metrics_nano: Dict[str, Any] = None,
    exact_metrics_55: Dict[str, Any] = None,
    fuzzy_metrics_55: Dict[str, Any] = None,
    exact_metrics_kimi: Dict[str, Any] = None,
    fuzzy_metrics_kimi: Dict[str, Any] = None,
    exact_metrics_nova: Dict[str, Any] = None,
    fuzzy_metrics_nova: Dict[str, Any] = None,
    exact_metrics_qwen: Dict[str, Any] = None,
    fuzzy_metrics_qwen: Dict[str, Any] = None,
) -> pd.DataFrame:
    """Build a compact model-wise comparison summary for Claude baseline."""
    data = [
        {
            "model": "Llama",
            "exact_accuracy_%": exact_metrics_llama["overall"]["accuracy"],
            "fuzzy_accuracy_%": fuzzy_metrics_llama["overall"]["accuracy"],
            "fuzzy_minus_exact_%": fuzzy_metrics_llama["overall"]["accuracy"] - exact_metrics_llama["overall"]["accuracy"],
            "records_analyzed": exact_metrics_llama["total_records"],
        },
        {
            "model": "Llama Scout",
            "exact_accuracy_%": exact_metrics_scout["overall"]["accuracy"],
            "fuzzy_accuracy_%": fuzzy_metrics_scout["overall"]["accuracy"],
            "fuzzy_minus_exact_%": fuzzy_metrics_scout["overall"]["accuracy"] - exact_metrics_scout["overall"]["accuracy"],
            "records_analyzed": exact_metrics_scout["total_records"],
        },
    ]

    if exact_metrics_nano is not None and fuzzy_metrics_nano is not None:
        data.append(
            {
                "model": "GPT Nano",
                "exact_accuracy_%": exact_metrics_nano["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_nano["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_nano["overall"]["accuracy"] - exact_metrics_nano["overall"]["accuracy"],
                "records_analyzed": exact_metrics_nano["total_records"],
            }
        )

    if exact_metrics_55 is not None and fuzzy_metrics_55 is not None:
        data.append(
            {
                "model": "GPT 5.5",
                "exact_accuracy_%": exact_metrics_55["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_55["overall"]["accuracy"],
                "fuzzy_minus_exact_%": (
                    fuzzy_metrics_55["overall"]["accuracy"]
                    - exact_metrics_55["overall"]["accuracy"]
                ),
                "records_analyzed": exact_metrics_55["total_records"],
            }
        )

    if exact_metrics_kimi is not None and fuzzy_metrics_kimi is not None:
        data.append(
            {
                "model": "Kimi-K2.5",
                "exact_accuracy_%": exact_metrics_kimi["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_kimi["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_kimi["overall"]["accuracy"] - exact_metrics_kimi["overall"]["accuracy"],
                "records_analyzed": exact_metrics_kimi["total_records"],
            }
        )

    if exact_metrics_nova is not None and fuzzy_metrics_nova is not None:
        data.append(
            {
                "model": "Amazon Nova Lite",
                "exact_accuracy_%": exact_metrics_nova["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_nova["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_nova["overall"]["accuracy"] - exact_metrics_nova["overall"]["accuracy"],
                "records_analyzed": exact_metrics_nova["total_records"],
            }
        )

    if exact_metrics_qwen is not None and fuzzy_metrics_qwen is not None:
        data.append(
            {
                "model": "Qwen 23.5B",
                "exact_accuracy_%": exact_metrics_qwen["overall"]["accuracy"],
                "fuzzy_accuracy_%": fuzzy_metrics_qwen["overall"]["accuracy"],
                "fuzzy_minus_exact_%": fuzzy_metrics_qwen["overall"]["accuracy"] - exact_metrics_qwen["overall"]["accuracy"],
                "records_analyzed": exact_metrics_qwen["total_records"],
            }
        )

    return pd.DataFrame(data)


def _print_metrics_block(exact_m: Dict, fuzzy_m: Dict, label: str):
    print("\n" + "=" * 60)
    print(f"EXACT MATCH METRICS - {label}")
    print("=" * 60)
    print(f"Overall Accuracy: {exact_m['overall']['accuracy']:.2f}%")
    print(f"Total Records Analyzed: {exact_m['total_records']}")
    print("\nField-by-Field Accuracy:")
    for field, acc in sorted(exact_m["field_metrics"].items(), key=lambda x: x[1], reverse=True):
        print(f"  {field:30s}: {acc:6.2f}%")

    print("\n" + "=" * 60)
    print(f"FUZZY MATCH METRICS - {label} (Threshold: {DEFAULT_FUZZY_THRESHOLD}%)")
    print("=" * 60)
    print(f"Overall Accuracy: {fuzzy_m['overall']['accuracy']:.2f}%")
    print(f"Total Records Analyzed: {fuzzy_m['total_records']}")
    print("\nField-by-Field Accuracy:")
    for field, acc in sorted(fuzzy_m["field_metrics"].items(), key=lambda x: x[1], reverse=True):
        print(f"  {field:30s}: {acc:6.2f}%")

    print("\n" + "=" * 60)
    print(f"COMPARISON: Fuzzy vs Exact - {label}")
    print("=" * 60)
    for field in exact_m["field_metrics"]:
        exact_acc = exact_m["field_metrics"].get(field, 0)
        fuzzy_acc = fuzzy_m["field_metrics"].get(field, 0)
        diff = fuzzy_acc - exact_acc
        print(f"  {field:30s}: Exact={exact_acc:6.2f}%, Fuzzy={fuzzy_acc:6.2f}% (d={diff:+6.2f}%)")


# Calculate metrics if comparison was successful
if "comparison_results" in locals():
    # GPT baseline: model vs GPT
    exact_metrics = calculate_accuracy_metrics(comparison_results["exact_match"], "exact")
    fuzzy_metrics = calculate_accuracy_metrics(comparison_results["fuzzy_match"], "fuzzy")
    _print_metrics_block(exact_metrics, fuzzy_metrics, "Llama vs GPT")

    exact_metrics_scout = calculate_accuracy_metrics(comparison_results["exact_match_scout"], "exact")
    fuzzy_metrics_scout = calculate_accuracy_metrics(comparison_results["fuzzy_match_scout"], "fuzzy")
    _print_metrics_block(exact_metrics_scout, fuzzy_metrics_scout, "Scout vs GPT")

    exact_metrics_nano = None
    fuzzy_metrics_nano = None
    if "exact_match_nano" in comparison_results and len(comparison_results["exact_match_nano"]) > 0:
        exact_metrics_nano = calculate_accuracy_metrics(comparison_results["exact_match_nano"], "exact")
        fuzzy_metrics_nano = calculate_accuracy_metrics(comparison_results["fuzzy_match_nano"], "fuzzy")
        _print_metrics_block(exact_metrics_nano, fuzzy_metrics_nano, "GPT Nano vs GPT")

    exact_metrics_55 = None
    fuzzy_metrics_55 = None

    if ("exact_match_gpt_55" in comparison_results and len(comparison_results["exact_match_gpt_55"]) > 0):
        exact_metrics_55 = calculate_accuracy_metrics(
            comparison_results["exact_match_gpt_55"],
            "exact"
        )

        fuzzy_metrics_55 = calculate_accuracy_metrics(
            comparison_results["fuzzy_match_gpt_55"],
            "fuzzy"
        )

        _print_metrics_block(
            exact_metrics_55,
            fuzzy_metrics_55,
            "GPT 5.5 vs GPT"
        )
    exact_metrics_kimi = None
    fuzzy_metrics_kimi = None
    if "exact_match_kimi" in comparison_results and len(comparison_results["exact_match_kimi"]) > 0:
        exact_metrics_kimi = calculate_accuracy_metrics(comparison_results["exact_match_kimi"], "exact")
        fuzzy_metrics_kimi = calculate_accuracy_metrics(comparison_results["fuzzy_match_kimi"], "fuzzy")
        _print_metrics_block(exact_metrics_kimi, fuzzy_metrics_kimi, "Kimi-K2.5 vs GPT")

    exact_metrics_claude = None
    fuzzy_metrics_claude = None
    if "exact_match_claude" in comparison_results and len(comparison_results["exact_match_claude"]) > 0:
        exact_metrics_claude = calculate_accuracy_metrics(comparison_results["exact_match_claude"], "exact")
        fuzzy_metrics_claude = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude"], "fuzzy")
        _print_metrics_block(exact_metrics_claude, fuzzy_metrics_claude, "Claude vs GPT")

    exact_metrics_nova = None
    fuzzy_metrics_nova = None
    if "exact_match_nova" in comparison_results and len(comparison_results["exact_match_nova"]) > 0:
        exact_metrics_nova = calculate_accuracy_metrics(comparison_results["exact_match_nova"], "exact")
        fuzzy_metrics_nova = calculate_accuracy_metrics(comparison_results["fuzzy_match_nova"], "fuzzy")
        _print_metrics_block(exact_metrics_nova, fuzzy_metrics_nova, "Amazon Nova Lite vs GPT")

    exact_metrics_qwen = None
    fuzzy_metrics_qwen = None
    if "exact_match_qwen" in comparison_results and len(comparison_results["exact_match_qwen"]) > 0:
        exact_metrics_qwen = calculate_accuracy_metrics(comparison_results["exact_match_qwen"], "exact")
        fuzzy_metrics_qwen = calculate_accuracy_metrics(comparison_results["fuzzy_match_qwen"], "fuzzy")
        _print_metrics_block(exact_metrics_qwen, fuzzy_metrics_qwen, "Qwen 23.5B vs GPT")

    model_wise_comparison_df = build_model_wise_summary(
        exact_metrics,
        fuzzy_metrics,
        exact_metrics_scout,
        fuzzy_metrics_scout,
        exact_metrics_nano,
        fuzzy_metrics_nano,
        exact_metrics_55,
        fuzzy_metrics_55,
        exact_metrics_kimi,
        fuzzy_metrics_kimi,
        exact_metrics_claude,
        fuzzy_metrics_claude,
        exact_metrics_nova,
        fuzzy_metrics_nova,
        exact_metrics_qwen,
        fuzzy_metrics_qwen,
    )
    print("\n" + "=" * 60)
    print("MODEL-WISE SUMMARY (GPT BASELINE)")
    print("=" * 60)
    print(model_wise_comparison_df.round(2).to_string(index=False))

    # Claude baseline: model vs Claude
    exact_metrics_claude_vs_llama = None
    fuzzy_metrics_claude_vs_llama = None
    exact_metrics_claude_vs_scout = None
    fuzzy_metrics_claude_vs_scout = None
    exact_metrics_claude_vs_nano = None
    fuzzy_metrics_claude_vs_nano = None
    exact_metrics_claude_vs_55 = None
    fuzzy_metrics_claude_vs_55 = None
    exact_metrics_claude_vs_kimi = None
    fuzzy_metrics_claude_vs_kimi = None
    exact_metrics_claude_vs_nova = None
    fuzzy_metrics_claude_vs_nova = None
    exact_metrics_claude_vs_qwen = None
    fuzzy_metrics_claude_vs_qwen = None

    if "exact_match_claude_vs_llama" in comparison_results and len(comparison_results["exact_match_claude_vs_llama"]) > 0:
        exact_metrics_claude_vs_llama = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_llama"], "exact")
        fuzzy_metrics_claude_vs_llama = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_llama"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_llama, fuzzy_metrics_claude_vs_llama, "Llama vs Claude")

    if "exact_match_claude_vs_scout" in comparison_results and len(comparison_results["exact_match_claude_vs_scout"]) > 0:
        exact_metrics_claude_vs_scout = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_scout"], "exact")
        fuzzy_metrics_claude_vs_scout = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_scout"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_scout, fuzzy_metrics_claude_vs_scout, "Llama Scout vs Claude")

    if "exact_match_claude_vs_nano" in comparison_results and len(comparison_results["exact_match_claude_vs_nano"]) > 0:
        exact_metrics_claude_vs_nano = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_nano"], "exact")
        fuzzy_metrics_claude_vs_nano = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_nano"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_nano, fuzzy_metrics_claude_vs_nano, "GPT Nano vs Claude")

    if (
    "exact_match_claude_vs_gpt_55" in comparison_results
    and len(comparison_results["exact_match_claude_vs_gpt_55"]) > 0
    ):

        exact_metrics_claude_vs_55 = calculate_accuracy_metrics(
            comparison_results["exact_match_claude_vs_gpt_55"],
            "exact"
        )

        fuzzy_metrics_claude_vs_55 = calculate_accuracy_metrics(
            comparison_results["fuzzy_match_claude_vs_gpt_55"],
            "fuzzy"
        )

        _print_metrics_block(
            exact_metrics_claude_vs_55,
            fuzzy_metrics_claude_vs_55,
            "GPT 5.5 vs Claude"
        )
    if "exact_match_claude_vs_kimi" in comparison_results and len(comparison_results["exact_match_claude_vs_kimi"]) > 0:
        exact_metrics_claude_vs_kimi = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_kimi"], "exact")
        fuzzy_metrics_claude_vs_kimi = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_kimi"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_kimi, fuzzy_metrics_claude_vs_kimi, "Kimi-K2.5 vs Claude")

    if "exact_match_claude_vs_nova" in comparison_results and len(comparison_results["exact_match_claude_vs_nova"]) > 0:
        exact_metrics_claude_vs_nova = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_nova"], "exact")
        fuzzy_metrics_claude_vs_nova = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_nova"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_nova, fuzzy_metrics_claude_vs_nova, "Amazon Nova Lite vs Claude")

    if "exact_match_claude_vs_qwen" in comparison_results and len(comparison_results["exact_match_claude_vs_qwen"]) > 0:
        exact_metrics_claude_vs_qwen = calculate_accuracy_metrics(comparison_results["exact_match_claude_vs_qwen"], "exact")
        fuzzy_metrics_claude_vs_qwen = calculate_accuracy_metrics(comparison_results["fuzzy_match_claude_vs_qwen"], "fuzzy")
        _print_metrics_block(exact_metrics_claude_vs_qwen, fuzzy_metrics_claude_vs_qwen, "Qwen 23.5B vs Claude")

    if exact_metrics_claude_vs_llama is not None and exact_metrics_claude_vs_scout is not None:
        claude_baseline_comparison_df = build_claude_baseline_summary(
        exact_metrics_claude_vs_llama,
        fuzzy_metrics_claude_vs_llama,
        exact_metrics_claude_vs_scout,
        fuzzy_metrics_claude_vs_scout,
        exact_metrics_claude_vs_nano,
        fuzzy_metrics_claude_vs_nano,
        exact_metrics_claude_vs_55,
        fuzzy_metrics_claude_vs_55,
        exact_metrics_claude_vs_kimi,
        fuzzy_metrics_claude_vs_kimi,
        exact_metrics_claude_vs_nova,
        fuzzy_metrics_claude_vs_nova,
        exact_metrics_claude_vs_qwen,
        fuzzy_metrics_claude_vs_qwen,
    )
        print("\n" + "=" * 60)
        print("MODEL-WISE SUMMARY (CLAUDE BASELINE)")
        print("=" * 60)
        print(claude_baseline_comparison_df.round(2).to_string(index=False))
else:
    print("⚠ Cannot calculate metrics - comparison not completed")


EXACT MATCH METRICS - Llama vs GPT
Overall Accuracy: 69.13%
Total Records Analyzed: 28

Field-by-Field Accuracy:
  taxonomy                      : 100.00%
  diagnosis_codes               : 100.00%
  tax_id                        :  96.43%
  invoice_date                  :  89.29%
  total_amount                  :  89.29%
  claimant_name                 :  82.14%
  date_of_service               :  78.57%
  charges                       :  78.57%
  cpt_codes                     :  67.86%
  claimant_number               :  60.71%
  units                         :  57.14%
  practice_address              :  35.71%
  billing_address               :  32.14%
  invoice_number                :   0.00%

FUZZY MATCH METRICS - Llama vs GPT (Threshold: 90%)
Overall Accuracy: 74.49%
Total Records Analyzed: 28

Field-by-Field Accuracy:
  taxonomy                      : 100.00%
  diagnosis_codes               : 100.00%
  tax_id                        :  96.43%
  total_amount                  :  92.86%

## Overall Results Report + Model Pricing

This section saves a text summary report with:
- GPT baseline model-wise summary
- Claude baseline model-wise summary (if available)
- Pricing reference for each model

In [86]:
# Save overall results + pricing to a text file
from pathlib import Path
from datetime import datetime

reports_dir = Path(r"C:\Users\Rohit.Sahu\Desktop\experiment\accuracy_reports")
reports_dir.mkdir(parents=True, exist_ok=True)

pricing_catalog_rows = [
    {
        "model": "Llama Maverick",
        "input_price_usd": 0.25,
        "input_unit": "per 1M tokens",
        "output_price_usd": 1.00,
        "output_unit": "per 1M tokens",
        "notes": "Provided by user",
    },
    {
        "model": "Llama Scout",
        "input_price_usd": 0.0002,
        "input_unit": "per 1K tokens",
        "output_price_usd": 0.00078,
        "output_unit": "per 1K tokens",
        "notes": "Provided by user",
    },
    {
        "model": "Kimi K2.5",
        "input_price_usd": 0.60,
        "input_unit": "per 1M tokens",
        "output_price_usd": "2.75-3.00",
        "output_unit": "per 1M tokens",
        "notes": "Output price is a range",
    },
    {
        "model": "Claude Haiku 3",
        "input_price_usd": 0.25,
        "input_unit": "per 1M tokens",
        "output_price_usd": 1.25,
        "output_unit": "per 1M tokens",
        "notes": "Provided by user",
    },
    {
        "model": "GPT-5.4 nano",
        "input_price_usd": 0.20,
        "input_unit": "per 1M tokens",
        "output_price_usd": 1.25,
        "output_unit": "per 1M tokens",
        "notes": "Provided by user",
    },
    {
        "model": "Amazon Nova 2 Lite",
        "input_price_usd": 0.30,
        "input_unit": "per 1M tokens",
        "output_price_usd": 2.50,
        "output_unit": "per 1M tokens",
        "notes": "Provided by user",
    },
    {
        "model": "Qwen3 235B A22B 2507",
        "input_price_usd": 0.22,
        "input_unit": "per 1M tokens",
        "output_price_usd": 0.88,
        "output_unit": "per 1M tokens",
        "notes": "Provided by user",
    },
]

# Map model labels used in comparison tables to pricing catalog names.
model_name_to_pricing = {
    "Llama": "Llama Maverick",
    "Llama Scout": "Llama Scout",
    "GPT Nano": "GPT-5.4 nano",
    "Kimi-K2.5": "Kimi K2.5",
    "Claude": "Claude Haiku 3",
    "Amazon Nova Lite": "Amazon Nova 2 Lite",
    "Amazon Nova 2 Lite": "Amazon Nova 2 Lite",
    "Qwen 23.5B": "Qwen3 235B A22B 2507",
    "Qwen3 235B A22B 2507": "Qwen3 235B A22B 2507",
}

used_pricing_models = set()
if "model_wise_comparison_df" in locals() and "model" in model_wise_comparison_df.columns:
    for model_name in model_wise_comparison_df["model"].dropna().astype(str).tolist():
        mapped = model_name_to_pricing.get(model_name)
        if mapped:
            used_pricing_models.add(mapped)

# If no summary is present, keep full catalog; otherwise keep only used models.
if used_pricing_models:
    pricing_rows = [row for row in pricing_catalog_rows if row["model"] in used_pricing_models]
else:
    pricing_rows = pricing_catalog_rows

pricing_df = pd.DataFrame(pricing_rows)

lines = []
lines.append("=" * 90)
lines.append("OCR MODEL EVALUATION - OVERALL RESULTS")
lines.append("=" * 90)
lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append("")

if "model_wise_comparison_df" in locals():
    lines.append("MODEL-WISE SUMMARY (GPT BASELINE)")
    lines.append("-" * 90)
    lines.append(model_wise_comparison_df.round(2).to_string(index=False))
    lines.append("")
else:
    lines.append("MODEL-WISE SUMMARY (GPT BASELINE)")
    lines.append("- Not available in current session")
    lines.append("")

if "claude_baseline_comparison_df" in locals():
    lines.append("MODEL-WISE SUMMARY (CLAUDE BASELINE)")
    lines.append("-" * 90)
    lines.append(claude_baseline_comparison_df.round(2).to_string(index=False))
    lines.append("")
else:
    lines.append("MODEL-WISE SUMMARY (CLAUDE BASELINE)")
    lines.append("- Not available in current session")
    lines.append("")

lines.append("MODEL PRICING REFERENCE")
lines.append("-" * 90)
lines.append(pricing_df.to_string(index=False))
lines.append("")
lines.append("Note: Pricing values are user-provided and may change by provider/region/time.")

overall_report_path = reports_dir / "overall_results_with_pricing.txt"
overall_report_path.write_text("\n".join(lines), encoding="utf-8")

print(f"Saved overall report to: {overall_report_path}")
print("\nPricing table:")
print(pricing_df.to_string(index=False))

if "model_wise_comparison_df" in locals():
    print("\nGPT baseline summary:")
    print(model_wise_comparison_df.round(2).to_string(index=False))

if "claude_baseline_comparison_df" in locals():
    print("\nClaude baseline summary:")
    print(claude_baseline_comparison_df.round(2).to_string(index=False))

Saved overall report to: C:\Users\Rohit.Sahu\Desktop\experiment\accuracy_reports\overall_results_with_pricing.txt

Pricing table:
               model  input_price_usd    input_unit output_price_usd   output_unit                   notes
      Llama Maverick           0.2500 per 1M tokens              1.0 per 1M tokens        Provided by user
         Llama Scout           0.0002 per 1K tokens          0.00078 per 1K tokens        Provided by user
           Kimi K2.5           0.6000 per 1M tokens        2.75-3.00 per 1M tokens Output price is a range
      Claude Haiku 3           0.2500 per 1M tokens             1.25 per 1M tokens        Provided by user
        GPT-5.4 nano           0.2000 per 1M tokens             1.25 per 1M tokens        Provided by user
  Amazon Nova 2 Lite           0.3000 per 1M tokens              2.5 per 1M tokens        Provided by user
Qwen3 235B A22B 2507           0.2200 per 1M tokens             0.88 per 1M tokens        Provided by user

GPT baseline 